# 우리 모델 그림 서버 (전시용)

앱에서 '그림 그리는 방법'을 **우리 모델**로 고르면, 기존 서버(`storybook_server`)가 동화 그림을 gpt-image 대신 이 노트북에 맡긴다.
이 노트북은 모델을 한 번 올려 두고, 맡겨진 동화를 한 권씩 그려 돌려준다 (그리는 코드는 `story_pipeline.py` - 시험 노트북과 같은 파일).

사용법
1. GPU T4 x2, Internet On 으로 **Run All** (모델 받기·올리기 약 2~3분)
2. ③ 셀이 출력하는 두 줄을 기존 서버의 `storybook_server/.env` 에 붙이고 서버를 다시 켠다 (IMAGE_ENGINE 은 넣지 않는다)
   `LOCAL_IMAGE_URL=https://....trycloudflare.com` / `LOCAL_IMAGE_TOKEN=...`
3. ④ 셀은 멈추지 않고 계속 돈다 (서버가 살아 있는 동안). 끝나면 세션 끄기(⏻)
4. (문제 볼 때) ④를 멈추고 ⑤를 실행하면 중간 그림(debug.zip)을 받는다. 서버는 그대로 켜져 있다

주의: 주소는 노트북을 켤 때마다 바뀐다. Kaggle 세션은 최대 12시간, GPU 는 주 30시간.
"Save Version"(백그라운드 실행)이 아니라 편집 화면에서 직접 실행해야 서버가 계속 살아 있다.

In [ ]:
# ① 설치 - 버전 고정 (2026-11-04 시험 노트북에서 성공한 조합) + 웹 서버(기존 서버와 같은 버전) + 터널 프로그램
!pip install -q diffusers==0.42.1 transformers==5.22.1 accelerate peft onnxruntime fastapi==0.135.3 uvicorn==0.44.0
!pip uninstall -y -q torchao
# Cloudflare 임시 터널: 계정 없이 https 주소를 하나 받아 이 노트북의 서버로 연결해 준다
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /kaggle/working/cloudflared

import importlib.metadata as md
TESTED = {"diffusers": "0.42.1", "transformers": "5.22.1", "torch": "2.11.0",
          "numpy": "2.1.3", "scipy": "1.16.3", "pillow": "13.0.0"}
for name, want in TESTED.items():
    have = md.version(name)
    print(f"{'  ' if have.startswith(want) else '⚠️'} {name} {have}" + ("" if have.startswith(want) else f" (시험한 버전 {want})"))
print("   (아직 고정 전)", ", ".join(f"{n} {md.version(n)}" for n in ("accelerate", "peft", "onnxruntime", "scikit-image")))

In [ ]:
%%writefile puppet.py
"""아이 그림 캐릭터를 몸통·팔다리로 나눠 자세를 바꾸는 도구.

학습 없이 처음 보는 그림에도 바로 쓴다. 그림의 선과 색은 그대로 두고 부분의 각도·위치만 바꾸므로 모양이 유지된다.

    sprite = cutout(drawing)              # 흰 바탕 그림 -> 투명 바탕 캐릭터
    rig = find_parts(sprite)              # 몸통에서 튀어나온 부분을 모두 후보로 (번호 붙임)
    overlay = numbered_overlay(sprite, rig)   # AI 에게 보여줄 번호 그림 (LIMB_QUESTION 과 함께)
    choose_limbs(rig, labels={"arms": [2, 3], "legs": []})   # AI 답 (또는 arms=2, legs=0 처럼 개수만)
    posed = pose(sprite, rig, "wave")     # 자세 적용 - 팔로 정해진 후보만 움직이고, 팔이 없으면 만들지 않는다
    name = pose_from_action("waving hello")   # 줄거리 문장 -> 자세 이름
"""
from dataclasses import dataclass, field, replace

import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from scipy import ndimage


# 원(반지름 r) 모양으로 깎기·불리기. 거리 계산으로 구하므로 비용이 r 과 상관없이 그림 크기에만 비례한다.
# (원 모양 도장을 찍는 방식은 r 의 제곱에 비례해 느려지고 메모리가 터진다 - 몸통이 큰 캐릭터일수록 r 이 커진다)
def _erode(mask, r):
    return ndimage.distance_transform_edt(mask) > r


def _dilate(mask, r):
    return ndimage.distance_transform_edt(~mask) <= r


def _open(mask, r):
    return _dilate(_erode(mask, r), r)


def _close(mask, r):
    return _erode(_dilate(mask, r), r)


# ── 1. 캐릭터 오려내기 ──────────────────────────────────────────────

def silhouette(img, tol=60):
    """단색 바탕 위 캐릭터 모양 (흰색 = 캐릭터).
    모서리 색과 비슷하고 가장자리와 이어진 곳만 바탕으로 보므로, 하얀 캐릭터의 윤곽선 안쪽도 캐릭터로 잡힌다."""
    a = np.array(img.convert("RGB").filter(ImageFilter.GaussianBlur(2))).astype(int)
    h, w = a.shape[:2]
    bg_color = np.median(np.array([a[2, 2], a[2, w - 3], a[h - 3, 2], a[h - 3, w - 3]]), axis=0)
    drawn = np.abs(a - bg_color).sum(axis=2) >= tol

    def enclosed(d):
        """가장자리와 이어진 바탕을 뺀 나머지 = 캐릭터."""
        background, _ = ndimage.label(~d)
        border = set(np.unique(np.concatenate([background[0], background[-1], background[:, 0], background[:, -1]])))
        return ~np.isin(background, list(border - {0})) | d

    plain = enclosed(drawn)
    closed = enclosed(_close(drawn, 4))       # 끊긴 선의 작은 틈을 메운 경우
    # 틈 메우기로 새로 생긴 부분 중 넓은 면(윤곽선이 살짝 끊긴 몸 안쪽)만 살리고,
    # 가는 틈(나란히 그린 두 다리 사이 등)은 다시 비워서 다리가 한 덩어리로 붙지 않게 한다
    added = closed & ~plain
    inside = plain | _open(added, 4)
    return Image.fromarray((inside * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(3))


def cutout(img, max_side=768, matte=None, inside=None):
    """흰(단색) 바탕 그림 -> 캐릭터만 남긴 투명 바탕 RGBA (여백 잘라냄).
    matte: 학습된 배경 제거 모델이 낸 모양(L). 주면 그것을 쓴다 - 바탕이 단색이 아니어도(그림자·번짐·색 바탕) 오려낸다.
    윤곽선으로 둘러싸인 하얀 몸 안쪽을 모델이 바탕으로 볼 수 있으므로 닫힌 구멍은 메운다.
    inside: 아이 그림 기준 몸 안쪽 (child_regions). 생성 그림의 선이 한 곳이라도 끊기면 하얀 안쪽이 바깥과 이어져
    구멍 메우기로도 안 채워지고 투명하게 잘린다 (별 친구 안쪽·주인공 얼굴이 비친 원인). 아이 그림에서는 선이 이어져 있으므로
    그 안쪽을 더한 뒤, 둘 사이에 남는 가는 틈만 닫고 구멍을 메운다.
    페이지에 들어갈 때는 어차피 이보다 작게 놓이므로, 긴 변을 max_side 로 줄여 이후 분석·자세 비용을 일정하게 한다."""
    if matte is None:
        shape = silhouette(img)
    else:
        m = np.array(matte.convert("L").resize(img.size)) > 128
        if inside is not None:
            m = _close(m | inside, max(img.size) * 0.004)
        m = ndimage.binary_fill_holes(m)
        shape = Image.fromarray((m * 255).astype(np.uint8))
    sprite = img.convert("RGBA")
    sprite.putalpha(shape)
    sprite = sprite.crop(shape.getbbox())
    sprite.thumbnail((max_side, max_side), Image.LANCZOS)
    return sprite


def _enclosed_by(lines):
    """선 그림에서 선으로 둘러싸인 곳 (가장자리와 이어진 바깥과 선 자체는 뺀다)."""
    labels, _ = ndimage.label(~lines)
    border = set(np.unique(np.concatenate([labels[0], labels[-1], labels[:, 0], labels[:, -1]]))) - {0}
    return ~np.isin(labels, list(border)) & ~lines


def _rooms_behind_doors(lines, doors=(0.03, 0.045, 0.07, 0.10, 0.13, 0.16)):
    """선으로 완전히 닫히지는 않았지만 몸의 일부인 곳: '좁은 입구를 지나야 들어가는 넓은 공간'이면서 '안에 아이가 그린 것이 있는' 곳.
    몸을 다 그리지 않아 목 아래가 열린 얼굴처럼, 아이들이 자주 선을 다 닫지 않는다 (닫힌 곳만 안쪽으로 보면 얼굴이 투명해진다).
    반대로 막대 다리 사이·들어 올린 팔과 머리 사이 같은 빈 틈은 안에 그린 것이 없으므로 바깥으로 둔다.
    입구 폭은 그림마다 달라서 여러 폭(그림 크기의 doors x2)으로 깎아 보고, 어느 하나에서라도 조건을 만족하면 안쪽으로 본다
    (깎는 폭이 공간보다 크면 공간 자체가 사라지고, 입구보다 작으면 입구가 안 끊긴다).
    작은 폭(3%, 4.5%)은 2026-11-11 앱 시험에서 추가: 흰둥이 머리는 귀 밑이 6% 열렸는데, 머리 안의 눈·입 선 때문에
    빈 공간이 좁아져 7% 이상으로 깎으면 머리 공간이 통째로 사라졌다 (머리가 투명하게 오려짐). 기존 시험 그림 13개는 변화 없음."""
    from skimage.segmentation import watershed
    size = lines.shape[0]
    labels, _ = ndimage.label(~lines)
    border = set(np.unique(np.concatenate([labels[0], labels[-1], labels[:, 0], labels[:, -1]]))) - {0}
    outside = np.isin(labels, list(border))
    found = np.zeros_like(lines)
    for door in doors:
        r = size * door
        core, n = ndimage.label(_open(outside, r))       # 입구가 2r 보다 좁으면 여기서 끊긴다
        edge = set(np.unique(np.concatenate([core[0], core[-1], core[:, 0], core[:, -1]]))) - {0}
        if n <= len(edge):
            continue
        # 깎은 부분을 되돌릴 때는 공간을 입구의 가장 좁은 곳에서 나눈다 (선에서 먼 곳부터 물이 차오르는 경계 = 워터셰드).
        # 그냥 r 만큼 불리면 입구 밖으로 넘치고, 바깥에서 r 안쪽을 빼면 공간 안쪽까지 깎인다.
        parts = watershed(-ndimage.distance_transform_edt(outside), core, mask=outside)
        for i in range(1, n + 1):
            if i in edge:
                continue
            room = ndimage.binary_fill_holes(parts == i)                          # 안의 선(눈·코·입)까지
            if (room & lines).sum() > size * size * 0.0005:                       # 안에 그린 것(눈·코·입 등)이 있어야
                # 입구에서 나눈 경계 너머의 통로(목선 사이)도 몸이므로, 선을 넘지 않고 통로를 따라 r 만큼 더 넓힌다
                found |= ndimage.binary_dilation(room, iterations=int(r), mask=outside | room)
    if found.any():
        # 열린 통로(목 사이)가 선이 끝난 곳 너머까지 이어지지 않게, 아이가 그린 선의 바깥 둘레(볼록 껍질) 안으로 자른다
        from skimage.morphology import convex_hull_image
        found &= convex_hull_image(lines)
    return found & ~lines


def child_regions(drawing, size):
    """아이 그림을 생성 그림과 같은 틀(size x size, scribble 과 같음)로 옮겨 잰 두 영역 (bool).
    inside: 아이가 선으로 둘러싼 몸 안쪽 - 선 굵기만큼 안으로 줄여, 생성 그림이 조금 어긋나도 선 바깥으로 넘치지 않게 한다.
    unpainted: 그 안쪽 중 아이가 칠하지 않고 종이로 둔 곳 - 다시 선에서 조금 더 떨어뜨린다 (선·눈 근처는 건드리지 않게)."""
    lines = np.array(to_scribble(drawing, size).convert("L")) > 128
    inside = _erode(_enclosed_by(lines) | _rooms_behind_doors(lines), size * 0.006)
    d = np.array(drawing.convert("RGB").resize((size, size))).astype(int)
    paper = (d.min(axis=2) > 215) & (d.max(axis=2) - d.min(axis=2) < 25)
    unpainted = _erode(inside & paper, size * 0.008)
    return inside, unpainted


def child_face(sheet, drawing, inside):
    """완성 그림(sheet)의 몸 안 표시(눈·코·입·안경·무늬)를 아이 그림의 몸 안 표시로 바꾼다.
    2026-11-11 앱 시험: 캐릭터 완성 단계에서 아이가 그리지 않은 코·웃는 입·안경이 생겼다 (흰둥이, 별 악당, 주인공 세모 코).
    완성 그림은 아이 그림 선(scribble)을 따라 같은 틀(1024)에 그려지므로 위치가 맞는다:
    ① 몸 안쪽(윤곽선에서 떨어진 곳)에서 몸 색보다 뚜렷이 진한 완성 그림의 표시를 주변 몸 색으로 지우고
    ② 같은 영역의 아이 그림 표시(몸 색보다 뚜렷이 진한 곳)를 아이 그림에서 그대로 옮긴다.
    윤곽선 근처는 건드리지 않는다 (윤곽은 완성 그림의 것을 쓴다). inside: child_regions 의 몸 안쪽 (1024)."""
    size = sheet.size[0]
    if not inside.any():
        return sheet
    # 몸 안쪽 = 아이 그림 전체 모양(몸을 가로지르는 선 포함)에서 바깥 윤곽만 띠만큼 뺀 곳.
    # inside 만 쓰면 몸을 가로지르는 아이 선(별의 모자 아래 선) 둘레가 빠져, 거기 걸친 안경이 남았다.
    lines = np.array(to_scribble(drawing, size).convert("L")) > 128
    figure = ndimage.binary_fill_holes(_close(lines | inside, size * 0.005))
    body = _erode(figure, size * 0.015)
    if body.sum() < size * size * 0.002:
        return sheet
    s = np.array(sheet.convert("RGB")).astype(np.float32)
    d = np.array(drawing.convert("RGB").resize((size, size), Image.LANCZOS)).astype(np.float32)
    s_lum, d_lum = s.mean(axis=2), d.mean(axis=2)
    s_marks = body & (s_lum < np.median(s_lum[body]) - 50)
    d_marks = body & (d_lum < np.median(d_lum[body]) - 50)
    # ① 완성 그림 표시 지우기: 표시를 뺀 몸 색을 부드럽게 퍼뜨려 채운다
    gone = _dilate(s_marks, 2) & body
    w = (body & ~gone).astype(np.float32)
    sigma = size * 0.01
    num = np.stack([ndimage.gaussian_filter(s[..., c] * w, sigma) for c in range(3)], axis=2)
    den = ndimage.gaussian_filter(w, sigma)[..., None]
    fill = np.where(den > 1e-3, num / np.maximum(den, 1e-3), np.median(s[body], axis=0))
    s[gone] = fill[gone]
    # ② 아이 표시 옮기기 (가장자리는 부드럽게)
    alpha = ndimage.gaussian_filter((_dilate(d_marks, 1) & body).astype(np.float32), 1.0)[..., None]
    s = s * (1 - alpha) + d * alpha
    return Image.fromarray(np.clip(s, 0, 255).astype(np.uint8))


def keep_unpainted(sheet, unpainted):
    """아이가 비워 둔 안쪽(종이)에 생성 그림이 색을 칠했으면 하얗게 되돌린다 (악당 얼굴의 연분홍).
    다시 그리기(캐릭터마다 수십 초) 대신 그린 뒤 고친다. 밝은 칠(최소 채널 140 이상)만 바꾸고 눈·입·선 같은 진한 부분은 둔다.
    경계는 흐리게 섞어 칠과 흰색 사이에 자국이 남지 않게 한다. 아이가 칠한 곳(종이가 아닌 곳)은 건드리지 않는다."""
    a = np.array(sheet.convert("RGB")).astype(np.float32)
    light = a.min(axis=2) > 140
    w = Image.fromarray(((unpainted & light) * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(max(sheet.size) / 400))
    w = np.array(w).astype(np.float32)[..., None] / 255 * light[..., None]
    white = np.array([253, 253, 253], np.float32)
    return Image.fromarray(np.clip(a * (1 - w) + white * w, 0, 255).astype(np.uint8))


ISNET_URL = "https://github.com/danielgatis/rembg/releases/download/v0.0.0/isnet-general-use.onnx"
# GitHub 이 안 될 때 받을 Hugging Face 복사본 (rembg 공식은 아님). 2026-11-08 GitHub 504 장애 때 확인:
# 따로 올린 두 복사본(fofr/comfyui, martintomov/comfy)의 크기·SHA256 이 같았다. 지문이 맞을 때만 쓴다.
ISNET_MIRROR = "https://huggingface.co/fofr/comfyui/resolve/main/rembg/isnet-general-use.onnx"
ISNET_SHA256 = "60920e99c45464f2ba57bee2ad08c919a52bbf852739e96947fbb4358c0d964a"


class Matte:
    """학습된 배경 제거 모델 (IS-Net, rembg 가 쓰는 것과 같은 ONNX 파일). 그림 -> 눈에 띄는 물체 모양(L).
    rembg 패키지는 numpy 를 올려 버전 고정을 깨므로 onnxruntime 으로 직접 돌린다 (CPU, 1장 몇 초).
    배경 후보에서는 '눈에 띄는 물체가 어디 있나'를 재는 데도 쓴다."""

    def __init__(self, path):
        import onnxruntime as ort
        self.sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
        self.name = self.sess.get_inputs()[0].name

    def __call__(self, img):
        x = np.array(img.convert("RGB").resize((1024, 1024), Image.LANCZOS)).astype(np.float32)
        x = x / max(float(x.max()), 1e-6) - 0.5
        out = self.sess.run(None, {self.name: x.transpose(2, 0, 1)[None]})[0][0, 0]
        out = (out - out.min()) / max(float(out.max() - out.min()), 1e-6)
        return Image.fromarray((out * 255).astype(np.uint8)).resize(img.size, Image.BILINEAR)


def cutout_problems(sprite):
    """오려낸 결과가 '그림 자체'가 아니라 포장까지 딸려 온 경우를 찾는다 (빈 목록이면 통과).
    - card: 모양이 자기 상자의 네 변 중 셋 이상을 거의 다 채운다 = 바탕 카드·액자째 오려냄
    - band: 윤곽을 따라 하얀 층이 둘러 있는데 몸 안쪽은 하얗지 않다 = 스티커 칼선 같은 흰 테두리.
      흰 층 바깥에 가는 선이 한 줄 더 있어도(가장 바깥이 흰색이 아니어도) 잡는다.
      몸 자체가 하얀 캐릭터(하얀 유령)는 안쪽도 하얘서 걸리지 않는다."""
    alpha = np.array(sprite.getchannel("A")) > 128
    if not alpha.any():
        return ["empty"]
    ys, xs = np.nonzero(alpha)
    box = alpha[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    k = max(1, int(max(box.shape) * 0.01))                         # 상자 변에서 살짝 안쪽 줄
    sides = [box[k].mean(), box[-1 - k].mean(), box[:, k].mean(), box[:, -1 - k].mean()]
    problems = ["card"] if sum(s > 0.7 for s in sides) >= 3 else []    # 한 변은 잘림·여백으로 빌 수 있다
    rgb = np.array(sprite.convert("RGB")).astype(int)
    light = (rgb.min(axis=2) > 215) & (rgb.max(axis=2) - rgb.min(axis=2) < 25)
    depth = ndimage.distance_transform_edt(alpha) / max(box.shape)      # 윤곽에서 안쪽으로 들어간 정도 (크기 대비)
    inner = alpha & (depth > 0.06)
    layers = [alpha & (depth > a) & (depth <= a + 0.005) for a in np.arange(0, 0.04, 0.005)]
    white_layer = max((light[l].mean() for l in layers if l.sum() > 20), default=0)
    # 윤곽 근처 어느 층이 둘레의 1/4 이상 하얗고, 몸 안쪽보다 4배 이상 하얗다 = 몸 색과 상관없는 흰 테두리.
    # (지난 시험들: 악당 흰 띠 0.33~0.66 / 안쪽 0.05 이하, 하얀 주인공 0.98 / 안쪽 0.95)
    if inner.any() and white_layer >= 0.25 and white_layer >= 4 * light[inner].mean():
        problems.append("band")
    return problems


def background_clutter(img, matte):
    """흰 바탕에 혼자 그리라고 한 그림에서, 물체 바깥에 다른 것이 얼마나 그려졌나 (0 = 바탕 한 색, 1 = 전부 다른 색).
    그림체 참고 그림(장면)이 너무 세게 걸리면 풍경·풀밭·액자까지 따라 그려지는데, 그러면 오려낸 물체에 풍경 조각이 붙고
    물체도 풍경에 섞여 흐려진다. 물체에서 조금 떨어진 바깥 픽셀 중 바탕색(바깥의 가운데 색)과 뚜렷이 다른 비율로 잰다.
    수채화 번짐처럼 부드러운 풍경은 잔무늬가 없어서, 잔무늬 대신 색 차이로 본다."""
    obj = np.array(matte.convert("L").resize(img.size)) > 128
    outside = ~_dilate(obj, max(img.size) * 0.02)
    if outside.sum() < 100:
        return 0.0
    a = np.array(img.convert("RGB")).astype(int)[outside]
    return float((np.abs(a - np.median(a, axis=0)).sum(axis=1) > 40).mean())


_COLOR_NAMES = {"black": (20, 20, 20), "dark gray": (90, 90, 90), "gray": (150, 150, 150), "brown": (110, 70, 40),
                "dark brown": (70, 40, 20), "red": (210, 40, 40), "dark red": (130, 20, 30), "orange": (240, 140, 40),
                "golden yellow": (210, 160, 30), "yellow": (250, 220, 60), "green": (60, 160, 60),
                "dark green": (30, 90, 40), "blue": (50, 100, 220), "navy blue": (30, 40, 110), "purple": (130, 60, 180),
                "pink": (240, 130, 180)}


def outline_color(drawing):
    """아이 그림의 바깥 윤곽선 색 이름 (윤곽선이 없으면 None).
    scribble 은 선이 '어디에' 있는지만 알려 주고, 선의 색·굵기는 IP-Adapter 가 아이 그림을 보고 전한다.
    칠한 면 가장자리에 붙은 윤곽선(주황 몸의 갈색 테두리)은 그것만으로는 약하게 전해져 그냥 색 경계로 그려지므로,
    색을 재서 글로도 알려 준다. 바깥 테두리 띠의 색이 몸 안쪽 색과 뚜렷이 다를 때만 윤곽선으로 본다."""
    sprite = cutout(drawing)
    alpha = np.array(sprite.getchannel("A")) > 128
    if not alpha.any():
        return None
    rgb = np.array(sprite.convert("RGB")).astype(int)
    depth = ndimage.distance_transform_edt(alpha) / max(alpha.shape)
    # 오려낸 가장자리에 남는 1~2픽셀 종이(흰색) 띠는 선 색이 아니므로 뺀다
    paper = (rgb.min(axis=2) > 215) & (rgb.max(axis=2) - rgb.min(axis=2) < 25)
    ring, inner = alpha & (depth <= 0.015) & ~paper, alpha & (depth > 0.04)
    if ring.sum() < 20 or not inner.any():
        return None
    line, fill = np.median(rgb[ring], axis=0), np.median(rgb[inner], axis=0)
    if np.abs(line - fill).sum() < 120:
        return None
    return min(_COLOR_NAMES, key=lambda n: np.abs(np.array(_COLOR_NAMES[n]) - line).sum())


def to_scribble(img, size=1024, edge_tol=40):
    """아이 그림 -> ControlNet scribble 조건 (검은 바탕 흰 선).
    선은 선 그대로 꽉 찬 한 줄로, 칠한 면은 바깥 윤곽선(색 경계)만 남긴다.
    색 경계만 선으로 쓰면 굵은 선 하나가 양쪽 경계 두 줄이 되어 속 빈 이중선(관 같은 팔)으로 그려지기 때문이다.
    색 경계로 나뉜 영역마다 두께를 재서, 캐릭터 전체 크기의 5% 보다 가는 영역은 선(의 안쪽)으로 채운다.
    펜·크레용 선은 캐릭터 크기에 비해 늘 가늘고, 칠한 몸·머리는 늘 굵기 때문에 그림 크기·선 굵기와 상관없이 나뉜다.
    (눈·단추처럼 작게 칠한 점도 꽉 채워지는데, 작은 점은 그게 원래 모양이다.)
    단, 종이와 같은 색인 영역(흰 눈알, 이빨, 줄무늬 사이 빈칸)은 가늘어도 펜 선일 수 없으므로 채우지 않는다.
    마지막으로 모든 선을 뼈대로 줄여 같은 굵기로 다시 그린다 (펜 굵기와 상관없이 일정)."""
    rgb = img.convert("RGB").filter(ImageFilter.GaussianBlur(1))
    edges = np.max([np.array(ch.filter(ImageFilter.FIND_EDGES)) for ch in rgb.split()], axis=0) > edge_tol
    edges[[0, -1]] = edges[:, [0, -1]] = False                     # FIND_EDGES 가 그림 가장자리에 내는 가짜 경계

    ys, xs = np.nonzero(edges)
    if not len(ys):
        return Image.new("RGB", (size, size), "black")
    figure = max(ys.max() - ys.min(), xs.max() - xs.min())        # 그린 캐릭터 전체 크기

    # 색 경계는 뾰족한 모서리에서 몇 픽셀씩 끊긴다 -> 끊긴 틈으로 선 안쪽이 옆의 넓은 면과 이어져 면으로 잘못 잡힌다.
    # 경계를 2픽셀 불려 틈을 막고 나눈 뒤, 선으로 정한 영역은 다시 2픽셀 불려 채운다.
    labels, n = ndimage.label(~_dilate(edges, 2))
    border = set(np.unique(np.concatenate([labels[0], labels[-1], labels[:, 0], labels[:, -1]]))) - {0}
    a = np.array(rgb).astype(int)
    paper = np.median(a[np.isin(labels, list(border))], axis=0) if border else np.array([255, 255, 255])
    strokes = np.zeros_like(edges)
    for i, sl in enumerate(ndimage.find_objects(labels), start=1):
        if i in border:                                            # 종이(바탕)
            continue
        region = labels[sl] == i
        if np.abs(np.median(a[sl][region], axis=0) - paper).sum() < 60:   # 종이 색 = 칠하지 않은 곳
            continue
        thick = 2 * ndimage.distance_transform_edt(np.pad(region, 1))[1:-1, 1:-1].max()
        if thick < figure * 0.05:                                  # 가는 영역 = 선의 안쪽
            strokes[sl] |= region
    scribble = edges | _dilate(strokes, 2.5)

    # 선 굵기를 아이 그림의 펜 굵기와 상관없이 일정하게: 선의 뼈대(가운데 한 줄)를 뽑아 같은 굵기로 다시 그린다.
    # 굵은 scribble 선은 ControlNet 이 '흰 띠 + 양쪽 가는 선'으로 그려 캐릭터 둘레에 흰 테두리가 생긴다 (굵은 크레용·마커일수록 심함).
    # 두께가 가장 두꺼운 점들(거리 극대점)로 가운데 줄을 잡으면 모서리·끝에 가지가 생겨, 정식 뼈대 추출(thinning)을 쓴다.
    from skimage.morphology import skeletonize
    # 색 경계선과 선 안쪽 사이의 1~2픽셀 틈은 뼈대에 작은 고리(선 끝·이음매의 혹)를 만든다 -> 그만큼만 메운다
    center = skeletonize(_close(scribble, 2.5))
    out = Image.fromarray((center * 255).astype(np.uint8)).resize((size, size), Image.BILINEAR)
    k = max(3, size // 150 | 1)                                    # 조건 그림 크기에 맞춘 일정한 굵기 (1024 -> 7)
    return out.point(lambda v: 255 if v > 0 else 0).filter(ImageFilter.MaxFilter(k)).convert("RGB")


# ── 2. 몸통과 팔다리 찾기 ──────────────────────────────────────────
# 모양 분석은 "몸통에서 튀어나온 부분"을 두께와 상관없이 모두 후보로 찾고 번호를 붙인다.
# 그중 무엇이 팔·다리인지는 그림을 이해하는 AI(지금 앱의 캐릭터 설명서 단계)가 정한다.
# 별 꼭짓점·귀·뿔처럼 모양만으로는 팔과 구분할 수 없는 부분이 있기 때문이다.

@dataclass
class Limb:
    id: int
    mask: np.ndarray        # 이 부분의 픽셀 (bool, 스프라이트 크기)
    pivot: tuple            # 몸통에 붙은 점 (x, y) - 여기를 축으로 돌린다
    kind: str               # 위치로 추정한 종류: "arm" | "leg" | "top" (AI 판단으로 바뀔 수 있음)
    side: str               # "left" | "right" | "center"
    movable: bool = False   # AI 가 팔·다리라고 한 후보만 True


@dataclass
class Rig:
    body: np.ndarray
    limbs: list = field(default_factory=list)
    line_color: tuple = (0, 0, 0)
    line_width: float = 4.0

    @property
    def moving(self):
        return [l for l in self.limbs if l.movable]




def find_parts(sprite):
    """몸통 = 가장 두꺼운 덩어리, 후보 = 몸통보다 가는 돌출부(선 한 줄이든 두껍게 칠한 팔이든).
    몸통 두께의 절반보다 가는 부분을 지우는 열림 연산으로 나누므로 그림 크기·선 굵기와 상관없다."""
    alpha = np.array(sprite.getchannel("A")) > 128
    h, w = alpha.shape
    rgb = np.array(sprite.convert("RGB")).astype(int)

    # 선 색·굵기: 팔을 떼어낸 어깨 자리에 윤곽선을 이어 그릴 때 쓴다
    ink = ~((rgb.sum(axis=2) > 600) & (rgb.max(axis=2) - rgb.min(axis=2) < 40)) & alpha   # 흰 번짐 제외
    ring = _dilate(alpha & ~_erode(alpha, 12), 2) & ink
    ring_px = rgb[ring] if ring.any() else rgb[alpha]
    lum = ring_px.sum(axis=1)
    line_color = np.median(ring_px[lum <= np.median(lum)], axis=0)
    line = ink & (np.abs(rgb - line_color).sum(axis=2) < 120)
    line_width = 2 * np.percentile(ndimage.distance_transform_edt(line)[line], 95) if line.any() else 4.0

    torso = ndimage.distance_transform_edt(alpha).max()          # 몸통 반지름 정도
    core = _open(alpha, max(2, torso * 0.5))
    labels, n = ndimage.label(core)
    if n == 0:
        return Rig(body=alpha, line_color=tuple(line_color), line_width=line_width)
    core = labels == (int(np.argmax(ndimage.sum(core, labels, range(1, n + 1)))) + 1)
    body = _dilate(core, 2) & alpha
    body &= ~_joined_legs(body)

    by, bx = np.nonzero(body)
    top, bottom, cx = by.min(), by.max(), bx.mean()
    body_h = bottom - top + 1
    near_body = _dilate(body, 4)
    dist_to_body = ndimage.distance_transform_edt(~body)

    labels, n = ndimage.label(alpha & ~body)
    found = []
    for i in range(1, n + 1):
        part = labels == i
        # 몸통 가장자리 부스러기는 몸통으로. 넓이만 보면 막대 선으로 그린 팔(길지만 픽셀 수가 적음)이 부스러기로 잘못 합쳐진다
        # (2026-11-08 Kaggle: 배경 제거 모델로 오려 선이 가늘어지자 주인공·악당 팔이 모두 사라져 자세·행동이 안 먹음).
        # 작은 조각이라도 몸통에서 멀리(몸통 키의 8% 이상) 뻗어 나가면 후보로 둔다.
        # (선 굵기는 기준으로 쓰지 않는다: 윤곽선 없이 칠한 몸은 몸 전체가 '선'으로 재여 굵기가 몸통만큼 나온다)
        # 몸통과 떨어진 작은 점(그림 구석의 잡티)은 팔이 아니므로, 작은 조각은 몸통 가까이(키의 5% 안)에서 시작해야 한다.
        if part.sum() < alpha.sum() * 0.008:
            near = dist_to_body[part]
            if near.max() < body_h * 0.08 or near.min() > body_h * 0.05:
                body |= part
                continue
        py, px = np.nonzero(part)
        contact = part & near_body
        if contact.any():
            cy_c, cx_c = np.nonzero(contact)
            pivot = (float(cx_c.mean()), float(cy_c.mean()))
        else:                                     # 몸통과 떨어져 그린 부분: 몸통에 가장 가까운 점을 축으로
            k = dist_to_body[py, px].argmin()
            pivot = (float(px[k]), float(py[k]))
        dy = py.mean() - pivot[1]
        rel_y = (pivot[1] - top) / body_h         # 몸통의 어느 높이에 붙었나 (0 = 맨 위, 1 = 맨 아래)
        side = "left" if px.mean() < cx - w * 0.05 else "right" if px.mean() > cx + w * 0.05 else "center"
        if (rel_y > 0.75 and dy > 0) or py.mean() > bottom:
            kind = "leg"
        elif (rel_y < 0.3 and dy < 0) or py.mean() < top:
            kind = "top"
        else:
            kind = "arm"
        found.append((py.mean(), px.mean(), part, pivot, kind, side))

    found.sort(key=lambda f: (round(f[0] / (h / 4)), f[1]))        # 위→아래, 왼→오른 순으로 번호
    limbs = [Limb(i + 1, part, pivot, kind, side) for i, (_, _, part, pivot, kind, side) in enumerate(found)]
    return Rig(body=body, limbs=limbs, line_color=tuple(int(v) for v in line_color), line_width=float(line_width))


def _joined_legs(body):
    """붙여 그린 두 다리: 몸통만큼 두꺼운 다리 덩어리는 '가장 두꺼운 덩어리'인 몸통에 흡수된다.
    몸통 아래 윤곽에서 아래로 열린 가장 깊은 홈(사타구니)을 찾아, 그 높이 아래를 홈 왼쪽·오른쪽으로 떼어 낸다.
    떼어 낸 부분은 다른 돌출부처럼 후보가 되고, 정말 다리인지는 AI 판단(또는 설명서의 다리 개수)이 정한다.
    홈 깊이가 몸통 키의 5% 보다 얕으면 그냥 둥근 몸 아래의 굴곡으로 본다."""
    rows = np.arange(body.shape[0])[:, None]
    has = body.any(axis=0)
    xs = np.nonzero(has)[0]
    none = np.zeros_like(body)
    if len(xs) < 5:
        return none
    bottom = np.where(body, rows, -1).max(axis=0)[xs].astype(float)    # 열마다 몸통 맨 아래
    # 홈 깊이 = 양쪽에서 가장 낮게 내려온 곳 중 덜 내려온 쪽 - 이 열의 바닥 (물을 부으면 고이는 깊이)
    left = np.maximum.accumulate(bottom)
    right = np.maximum.accumulate(bottom[::-1])[::-1]
    depth = np.minimum(left, right) - bottom
    k = int(depth.argmax())
    by = np.nonzero(body.any(axis=1))[0]
    if depth[k] < (by.max() - by.min()) * 0.05:
        return none
    apex_x, apex_y = xs[k], int(bottom[k])
    below = body & (rows > apex_y)
    cols = np.arange(body.shape[1])[None, :]
    legs = (below & (cols < apex_x)) | (below & (cols > apex_x))
    return legs


def choose_limbs(rig, arms=None, legs=None, labels=None):
    """움직일 후보를 정한다.
    labels: AI 가 번호로 답한 경우 {"arms": [2, 3], "legs": [5, 6]}
    arms/legs: 설명서에 개수만 있는 경우 - 위치로 추정한 종류가 맞는 후보 중 큰 것부터 고른다
    둘 다 없으면 아무것도 움직이지 않는다 (팔을 만들지 않는다)."""
    for l in rig.limbs:
        l.movable = False
    if labels:
        for key, kind in (("arms", "arm"), ("legs", "leg")):
            for l in rig.limbs:
                if l.id in labels.get(key, []):
                    l.kind, l.movable = kind, True
        return rig
    for kind, count in (("arm", arms or 0), ("leg", legs or 0)):
        cands = sorted([l for l in rig.limbs if l.kind == kind], key=lambda l: -l.mask.sum())
        for l in cands[:count]:
            l.movable = True
    return rig


def analyze(sprite, arms=None, legs=None, labels=None):
    return choose_limbs(find_parts(sprite), arms, legs, labels)


def numbered_overlay(sprite, rig):
    """AI 에게 보여줄 그림: 후보마다 다른 색과 번호를 칠한다."""
    palette = [(230, 60, 60), (40, 150, 230), (40, 170, 70), (240, 160, 30), (160, 80, 210),
               (20, 180, 180), (230, 90, 170), (120, 120, 120)]
    base = Image.alpha_composite(Image.new("RGBA", sprite.size, "white"), sprite)
    over = np.array(base.convert("RGB"))
    for l in rig.limbs:
        over[l.mask] = (over[l.mask] * 0.4 + np.array(palette[(l.id - 1) % len(palette)]) * 0.6).astype(np.uint8)
    out = Image.fromarray(over)
    d = ImageDraw.Draw(out)
    fs = max(14, sprite.width // 14)
    for l in rig.limbs:
        ys, xs = np.nonzero(l.mask)
        x, y = xs.mean(), ys.mean()
        d.ellipse((x - fs * 0.7, y - fs * 0.7, x + fs * 0.7, y + fs * 0.7), fill="white", outline="black", width=2)
        d.text((x - fs * 0.25, y - fs * 0.45), str(l.id), fill="black")
    return out


# 지금 앱의 캐릭터 설명서(GPT 비전) 요청에 함께 넣을 질문. 두 번째 이미지로 numbered_overlay 를 보낸다.
LIMB_QUESTION = (
    "The second image shows the same drawing with numbered colored regions that stick out from the body. "
    "Which numbers are ARMS and which are LEGS that the child actually drew? Thick arms count as arms. "
    "Ears, horns, hair, star points, fins, tails, wings and decorations are NOT arms or legs. "
    'Answer only JSON like {"arms": [2, 3], "legs": [5, 6]}; use empty lists if none.'
)


# ── 3. 자세 ───────────────────────────────────────────────────────
# 처음 그린 각도와 상관없이 "어느 방향을 가리키게 할지"를 정한다 (그래서 이미 위로 뻗은 팔도 몸 뒤로 숨지 않는다).
# arm: 바깥쪽 수평선 기준 높이(도). 양수 = 위로, 음수 = 아래로.  leg: 바로 아래 기준 바깥으로 벌린 각도(도).
# tilt: 몸 전체 기울기(양수 = 왼쪽으로).  stretch: (가로, 세로) 배율.
# 지정하지 않은 팔다리는 그린 그대로 둔다. 팔다리가 없는 캐릭터는 tilt·stretch 만으로 표현된다.
POSES = {
    "stand":  {},
    "wave":   {"arm": {"right": 60}, "tilt": 4},
    "cheer":  {"arm": {"left": 65, "right": 65}, "stretch": (0.97, 1.05)},
    "walk":   {"arm": {"left": -55, "right": -25}, "leg": {"left": 22, "right": -6}, "tilt": -5},
    "jump":   {"arm": {"left": 40, "right": 40}, "leg": {"left": 8, "right": 8}, "stretch": (0.93, 1.08), "float": True},
    "sad":    {"arm": {"left": -75, "right": -75}, "tilt": -7, "stretch": (1.03, 0.95)},
    "scared": {"arm": {"left": 25, "right": 25}, "tilt": 8, "stretch": (0.92, 0.94)},
}

# 줄거리 문장 -> 자세. 앱에서는 언어 모델이 장면마다 이 목록 중 하나를 고르게 하면 된다.
_KEYWORDS = [
    ("wave", ["wave", "waving", "hello", "greet", "인사", "손을 흔", "손 흔"]),
    ("jump", ["jump", "leap", "hop", "bounce", "뛰어오", "점프", "폴짝", "뛰"]),
    ("cheer", ["cheer", "celebrat", "dance", "danc", "happy", "hooray", "춤", "기뻐", "환호", "만세"]),
    ("walk", ["walk", "run", "go ", "going", "stroll", "걷", "걸어", "달리", "갔"]),
    ("sad", ["sad", "cry", "tear", "lonely", "슬프", "울", "외로"]),
    ("scared", ["scared", "afraid", "fear", "surpris", "tremb", "무서", "놀라", "떨"]),
]


def pose_from_action(text):
    t = (text or "").lower()
    for name, words in _KEYWORDS:
        if any(w in t for w in words):
            return name
    return "stand"


def _limb_angle(limb, target):
    """지금 가리키는 방향에서 목표 방향까지 돌릴 각도(반시계 +). 화면 좌표는 y 가 아래로 커서 부호를 뒤집는다."""
    ys, xs = np.nonzero(limb.mask)
    current = np.degrees(np.arctan2(-(ys.mean() - limb.pivot[1]), xs.mean() - limb.pivot[0]))
    if limb.kind == "arm":
        goal = target if limb.side == "right" else 180 - target
    else:
        goal = -90 + target if limb.side == "right" else -90 - target
    return (goal - current + 180) % 360 - 180


def _hinge(limb, body, angle):
    """돌리는 축 = 몸통과 붙은 면에서 '접히는 쪽' 끝 (팔을 내리면 아래 모서리, 올리면 위 모서리).
    붙은 면 한가운데를 축으로 돌리면 굵은 팔일수록 반쪽이 몸통 안으로 쓸려 들어가고,
    몸통 가까이 거의 안 움직이는 팔 윤곽선 조각이 새 팔 모양 안쪽에 남는다 (흰 점·검은 자국).
    실제 어깨처럼 접히는 쪽 모서리를 축으로 하면 팔 굵기와 상관없이 몸 바깥으로 접힌다. 다리도 같다."""
    contact = limb.mask & _dilate(body, 4)
    if not contact.any() or abs(angle) < 1:
        return limb.pivot
    cy, cx = np.nonzero(contact)
    ly, lx = np.nonzero(limb.mask)
    ux, uy = lx.mean() - limb.pivot[0], -(ly.mean() - limb.pivot[1])     # 축 -> 팔 방향 (y 위로)
    nx, ny = -uy, ux                                                      # 그 왼쪽(반시계) 방향
    side = np.sign(angle) * ((cx - limb.pivot[0]) * nx + (-(cy - limb.pivot[1])) * ny)
    k = side.argmax()
    return (float(cx[k]), float(cy[k]))


def _smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def _bend(canvas, rig, moving, pad):
    """자르지 않고 팔·다리를 고무처럼 휘게 한다.
    몸통은 그대로 두고, 팔·다리 영역에 촘촘한 삼각형 그물을 씌워 꼭짓점을 옮긴다.
    몸통 경계에서는 전혀 움직이지 않고(0), 멀어질수록 목표 각도까지 돌아가므로(1)
    어깨가 늘어나며 휘어질 뿐 끊기거나 잘린 자국이 생기지 않는다."""
    W, H = canvas.size
    sh, sw = rig.body.shape
    rgba = np.array(canvas).astype(float) / 255
    rgba[..., :3] *= rgba[..., 3:4]                 # 알파를 곱해 두면 가장자리가 검게 번지지 않는다
    visible = rgba[..., 3] > 0

    still = np.zeros((H, W), bool)                 # 움직이지 않는 부분 (몸통 + 움직이지 않는 후보)
    still[pad:pad + sh, pad:pad + sw] = rig.body
    for l in rig.limbs:
        if not l.movable:
            still[pad:pad + sh, pad:pad + sw] |= l.mask
    d_out = ndimage.distance_transform_edt(~still)
    limb_d = []
    for limb, _ in moving:
        m = np.zeros((H, W), bool)
        m[pad:pad + sh, pad:pad + sw] = limb.mask
        limb_d.append(ndimage.distance_transform_edt(~m))
    limb_d = np.array(limb_d)
    nearest = limb_d.argmin(axis=0)

    body_layer = rgba.copy()
    limb_layers = []
    step = max(4, min(sw, sh) // 60)
    for k, (limb, angle) in enumerate(moving):
        # 이 팔다리의 몫: 움직이지 않는 부분보다 이 팔다리에 더 가까운 바깥 영역
        owned = ~still & (nearest == k) & (limb_d[k] < d_out)
        region = owned & _dilate(visible, 3)
        if not region.any():
            continue
        body_layer[owned] = 0
        thickness = 2 * ndimage.distance_transform_edt(limb.mask).max()
        ly, lx = np.nonzero(limb.mask)
        length = np.hypot(lx - limb.pivot[0], ly - limb.pivot[1]).max()
        # 어깨에서 휘어지는 구간의 길이. 팔 길이의 40% 를 넘으면 짧고 굵은 팔이 갈고리처럼 말리므로 제한한다
        band = min(max(thickness * 1.5, min(sw, sh) * 0.08), length * 0.4)
        weight = np.where(owned, _smoothstep(d_out / band), 0.0)
        # 몸통 층에서 지운 경계와 팔 층의 경계가 둘 다 반투명이면 겹쳐도 1픽셀 틈이 비친다.
        # 팔 층이 경계 너머 2픽셀을 더 가져가 몸통과 겹치게 한다 (경계 근처는 거의 움직이지 않아 자국이 없다)
        src_img = rgba * (owned | (_dilate(owned, 2) & still))[..., None]
        cx, cy = limb.pivot[0] + pad, limb.pivot[1] + pad
        th = np.radians(angle)

        ys, xs = np.nonzero(region)
        x0, x1 = xs.min() - step, xs.max() + step
        y0, y1 = ys.min() - step, ys.max() + step
        gx = np.arange(x0, x1 + step, step, dtype=float)
        gy = np.arange(y0, y1 + step, step, dtype=float)
        vx, vy = np.meshgrid(gx, gy)
        wv = ndimage.map_coordinates(weight, [vy, vx], order=1, mode="constant", cval=0.0)
        dx, dy = vx - cx, vy - cy
        rx = cx + dx * np.cos(th) + dy * np.sin(th)            # 화면에서 반시계 방향 회전
        ry = cy - dx * np.sin(th) + dy * np.cos(th)
        ux, uy = vx + wv * (rx - vx), vy + wv * (ry - vy)      # 옮긴 꼭짓점

        cell_has = ndimage.maximum_filter(region.astype(np.uint8), size=step * 2 + 1)
        layer = np.zeros_like(rgba)
        # 크게 접으면 굽는 안쪽이 바깥쪽보다 덜 돌아서 그물 삼각형이 뒤집혀 겹친다(접힘).
        # 뒤집힌 삼각형은 팔 아랫선·가장자리를 팔 안쪽에 다시 그리므로, 먼저 그리고 정상 삼각형으로 덮는다.
        # 접힌 부분은 늘 정상 면과 겹쳐 있으므로 덮으면 사라지고, 정상 면이 없는 곳만 뒤집힌 그림이 남아 빈틈을 메운다.
        normal, folded = [], []
        for i in range(len(gy) - 1):
            for j in range(len(gx) - 1):
                cyc, cxc = int(min(max(gy[i] + step / 2, 0), H - 1)), int(min(max(gx[j] + step / 2, 0), W - 1))
                if not cell_has[cyc, cxc]:
                    continue
                for tri in (((i, j), (i, j + 1), (i + 1, j)), ((i, j + 1), (i + 1, j + 1), (i + 1, j))):
                    src = [(vx[a, b], vy[a, b]) for a, b in tri]
                    dst = [(ux[a, b], uy[a, b]) for a, b in tri]
                    (normal if _signed_area(src) * _signed_area(dst) > 0 else folded).append((src, dst))
        for src, dst in folded + normal:
            _raster_triangle(layer, src_img, src, dst)
        limb_layers.append(layer)

    out = body_layer
    for layer in limb_layers:                      # 팔다리를 몸통 위에 얹는다 (알파를 곱한 상태의 over 합성)
        out = layer + out * (1 - layer[..., 3:4])
    alpha = out[..., 3:4]
    out[..., :3] = np.where(alpha > 1e-3, out[..., :3] / np.maximum(alpha, 1e-3), 0)
    return Image.fromarray((np.clip(out, 0, 1) * 255).astype(np.uint8), "RGBA")


def _signed_area(tri):
    (x1, y1), (x2, y2), (x3, y3) = tri
    return (x2 - x1) * (y3 - y1) - (x3 - x1) * (y2 - y1)


def _raster_triangle(dst_img, src_img, src_tri, dst_tri):
    """옮긴 삼각형(dst) 안의 픽셀마다 원래 삼각형(src)의 같은 자리를 찾아 색을 가져온다."""
    H, W = dst_img.shape[:2]
    (x1, y1), (x2, y2), (x3, y3) = dst_tri
    bx0, bx1 = int(max(np.floor(min(x1, x2, x3)), 0)), int(min(np.ceil(max(x1, x2, x3)), W - 1))
    by0, by1 = int(max(np.floor(min(y1, y2, y3)), 0)), int(min(np.ceil(max(y1, y2, y3)), H - 1))
    if bx1 < bx0 or by1 < by0:
        return
    det = (y2 - y3) * (x1 - x3) + (x3 - x2) * (y1 - y3)
    if abs(det) < 1e-6:
        return
    py, px = np.mgrid[by0:by1 + 1, bx0:bx1 + 1].astype(float)
    l1 = ((y2 - y3) * (px - x3) + (x3 - x2) * (py - y3)) / det
    l2 = ((y3 - y1) * (px - x3) + (x1 - x3) * (py - y3)) / det
    l3 = 1 - l1 - l2
    inside = (l1 >= -1e-3) & (l2 >= -1e-3) & (l3 >= -1e-3)
    if not inside.any():
        return
    (sx1, sy1), (sx2, sy2), (sx3, sy3) = src_tri
    sx = l1 * sx1 + l2 * sx2 + l3 * sx3
    sy = l1 * sy1 + l2 * sy2 + l3 * sy3
    iy, ix = py[inside].astype(int), px[inside].astype(int)
    for c in range(4):
        dst_img[iy, ix, c] = ndimage.map_coordinates(src_img[..., c], [sy[inside], sx[inside]], order=1, mode="constant")


def pose(sprite, rig, name):
    """자세를 적용한 RGBA 스프라이트를 돌려준다. 'float' 자세는 .info['float'] = True 로 표시."""
    spec = POSES.get(name, {})
    w, h = sprite.size
    pad = int(max(w, h) * 0.45)                    # 팔이 들려도 잘리지 않게 여유
    W, H = w + 2 * pad, h + 2 * pad

    # AI 가 팔·다리라고 한 후보만 움직인다. 축은 접히는 쪽 모서리(_hinge)로 옮긴 사본을 쓴다
    hinged = {}
    for limb in rig.limbs:
        if limb.movable and limb.side in spec.get(limb.kind, {}):
            angle = _limb_angle(limb, spec[limb.kind][limb.side])
            hinged[id(limb)] = (replace(limb, pivot=_hinge(limb, rig.body, angle)), angle)
    moving = list(hinged.values())
    canvas = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    canvas.paste(sprite, (pad, pad))
    if moving:
        canvas = _bend(canvas, rig, moving, pad)

    # 소품을 붙일 자리: 머리 꼭대기 = 몸통 맨 윗부분 (귀·뿔·머리카락 같은 후보는 몸통이 아니므로 빠진다),
    # 손 = 팔 후보에서 어깨(축)로부터 가장 먼 점. 그림과 똑같이 돌리고 늘이고 기울여 자세 적용 뒤 좌표로 옮긴다.
    # 몸통에 합쳐 둔 가장자리 부스러기가 머리로 잡히지 않게, 가장 큰 덩어리만 본다
    labels, n = ndimage.label(rig.body)
    core = labels == (np.argmax(ndimage.sum(rig.body, labels, range(1, n + 1))) + 1) if n > 1 else rig.body
    by, bx = np.nonzero(core)
    top = by.min()
    points = {"head": (float(bx[by <= top + max(2, (by.max() - top) * 0.03)].mean()) + pad, float(top) + pad)}
    for limb in rig.limbs:
        if limb.movable and limb.kind == "arm":
            ly, lx = np.nonzero(limb.mask)
            k = np.hypot(lx - limb.pivot[0], ly - limb.pivot[1]).argmax()
            p = (lx[k] + pad, ly[k] + pad)
            if id(limb) in hinged:                 # 팔 끝은 휘는 구간 밖이라 목표 각도만큼 그대로 돈다
                moved, angle = hinged[id(limb)]
                p = _rotate_point(p, (moved.pivot[0] + pad, moved.pivot[1] + pad), angle)
            points[f"hand_{limb.side}"] = p

    sx, sy = spec.get("stretch", (1.0, 1.0))
    if (sx, sy) != (1.0, 1.0):                     # 발바닥 기준으로 늘이거나 줄인다
        bbox = canvas.getbbox()
        part = canvas.crop(bbox)
        part = part.resize((max(1, int(part.width * sx)), max(1, int(part.height * sy))), Image.LANCZOS)
        canvas = Image.new("RGBA", (W, H), (0, 0, 0, 0))
        ox, oy = int((bbox[0] + bbox[2] - part.width) / 2), bbox[3] - part.height
        canvas.paste(part, (ox, oy), part)
        kx, ky = part.width / (bbox[2] - bbox[0]), part.height / (bbox[3] - bbox[1])
        points = {n: (ox + (x - bbox[0]) * kx, oy + (y - bbox[1]) * ky) for n, (x, y) in points.items()}
    if spec.get("tilt"):
        bbox = canvas.getbbox()
        center = ((bbox[0] + bbox[2]) / 2, bbox[3])
        canvas = canvas.rotate(spec["tilt"], center=center, resample=Image.BICUBIC)
        points = {n: _rotate_point(p, center, spec["tilt"]) for n, p in points.items()}

    bbox = canvas.getbbox()
    out = canvas.crop(bbox)
    out.info["float"] = bool(spec.get("float"))
    out.info["points"] = {n: (x - bbox[0], y - bbox[1]) for n, (x, y) in points.items()}
    return out


def _rotate_point(p, center, degrees):
    """화면에서 반시계 방향으로 degrees 만큼 돌린 점 (_bend 의 꼭짓점 회전, PIL rotate 와 같은 방향)."""
    th = np.radians(degrees)
    dx, dy = p[0] - center[0], p[1] - center[1]
    return (center[0] + dx * np.cos(th) + dy * np.sin(th), center[1] - dx * np.sin(th) + dy * np.cos(th))


def debug_view(sprite, rig):
    """분석 결과 확인용: 몸통 파랑, 움직이는 팔 빨강·다리 초록, 움직이지 않는 후보 회색, 축은 검은 점."""
    colors = {"arm": (230, 60, 60), "leg": (40, 170, 70)}
    a = np.zeros((*rig.body.shape, 3), np.uint8)
    a[rig.body] = (70, 120, 230)
    for l in rig.limbs:
        a[l.mask] = colors.get(l.kind, (150, 150, 150)) if l.movable else (170, 170, 170)
    img = Image.fromarray(a).convert("RGBA")
    img.putalpha(sprite.getchannel("A"))
    bg = Image.new("RGBA", img.size, "white")
    out = Image.alpha_composite(bg, img)
    d = ImageDraw.Draw(out)
    for l in rig.moving:
        x, y = l.pivot
        d.ellipse((x - 5, y - 5, x + 5, y + 5), fill="black")
    return out.convert("RGB")

In [ ]:
%%writefile scene_plan.py
"""장면 계획 -> 페이지 조립.

언어 모델이 줄거리를 읽고 페이지(칸)마다 "누가, 어디에, 어떤 자세로, 어떤 소품과" 나오는지 JSON 으로 정하면,
이 모듈이 배경 위에 캐릭터를 배치·자세 적용하고 소품과 조연을 붙여 페이지를 만든다.

계획을 받아 오는 지시문(GPT)은 기존 서버 storybook_server/services/local_image_engine.py 에 있다 (SLOTS·SIZES·POSES·ANCHORS 와
같은 이름을 쓴다). 이 모듈은 받은 계획을 정리(normalize_plan)하고 조립한다. 이미지 생성은 하지 않는다.
"""
import numpy as np
from PIL import Image, ImageChops, ImageDraw, ImageFilter, ImageOps
from scipy import ndimage

from puppet import POSES, pose

# ── 1. 계획 형식 ──────────────────────────────────────────────────

SLOTS = {"left": 0.18, "center-left": 0.34, "center": 0.5, "center-right": 0.66, "right": 0.82}
SIZES = {"small": 0.30, "medium": 0.44, "large": 0.58}          # 페이지 높이 대비 캐릭터 키
PLACES = {"ground": 0.90, "air": 0.50}                          # 발(아래쪽)이 놓이는 높이
ANCHORS = ("on_head", "in_hand", "beside", "above")

def normalize_plan(plan, asset_ids):
    """언어 모델 답에서 틀린 값은 기본값으로 바꾸고, 모르는 캐릭터·소품은 버린다 (조립이 멈추지 않게)."""
    pages = []
    for p in plan.get("pages", []):
        panels = []
        for panel in p.get("panels", []) or [p]:
            chars, seen = [], set()
            for c in panel.get("characters", [])[:3]:
                if c.get("id") not in asset_ids or c["id"] in seen:
                    continue
                seen.add(c["id"])
                chars.append({
                    "id": c["id"],
                    "slot": c.get("slot") if c.get("slot") in SLOTS else "center",
                    "size": c.get("size") if c.get("size") in SIZES else "medium",
                    "place": c.get("place") if c.get("place") in PLACES else "ground",
                    "pose": c.get("pose") if c.get("pose") in POSES else "stand",
                    "facing": c.get("facing") if c.get("facing") in ("left", "right") else "right",
                })
            props = [pr for pr in panel.get("props", [])[:2]
                     if pr.get("anchor") in ANCHORS and pr.get("owner") in seen and pr.get("name")]
            extras = [e for e in panel.get("extras", []) if e.get("name")]
            panels.append({"background": panel.get("background", "a cozy storybook place"),
                           "characters": chars, "props": props, "extras": extras})
        pages.append({"page": p.get("page", len(pages) + 1), "panels": panels})
    # 캐릭터가 쓰거나 드는 소품만 따로 그려 붙인다. 한 번이라도 들린 소품은 다른 칸에서도 같은 그림을 쓴다 (모양 일치).
    # 옆에 놓이기만 하는 소품은 배경과 함께 그린다 (painted).
    held = {pr["name"] for p in pages for panel in p["panels"] for pr in panel["props"] if pr["anchor"] in HELD}
    for p in pages:
        for panel in p["panels"]:
            for pr in panel["props"]:
                pr["painted"] = pr["name"] not in held
    return pages


HELD = ("on_head", "in_hand")
_SIDE_WORDS = [(0.26, "on the far left"), (0.42, "on the left"), (0.58, "in the middle"), (0.74, "on the right"),
               (1.01, "on the far right")]


def background_prompt(panel):
    """배경 그림 프롬프트: 한 번만 나오는 조연과 옆에 놓이는 소품은 배경에 같이 그리되,
    캐릭터가 들어갈 자리를 피해 가장 먼 빈 쪽에 작게 그리라고 적는다 (배경이 캐릭터 자리를 모르면 겹치기 때문)."""
    painted = ([e["name"] for e in panel["extras"] if not e.get("recurring")]
               + [pr["name"] for pr in panel["props"] if pr.get("painted")])
    if not painted:
        return panel["background"]
    # 피할 자리 = 아이 캐릭터 + 여러 번 나오는 조연이 놓일 자리. 그중 어느 것과도 가장 먼 가로 위치를 고른다
    recurring = sum(1 for e in panel["extras"] if e.get("recurring"))
    used = [SLOTS[c["slot"]] for c in panel["characters"]] + [SLOTS[s] for s in _free_slots(panel)[:recurring]]
    x = max(np.linspace(0.1, 0.9, 17), key=lambda s: min((abs(s - u) for u in used), default=1))
    where = next(w for edge, w in _SIDE_WORDS if x < edge)
    return f"{panel['background']}, with {', '.join(painted)} small in the distance {where}"

# ── 2. 페이지 조립 ────────────────────────────────────────────────

GOLD = (240, 190, 50)


def _layout(size, entries, sprites):
    """칸 안에서 각 캐릭터가 실제로 차지하는 영역(자세 적용 뒤 크기)으로 자리를 정한다 -> [(x, y, 배율)].
    슬롯은 '대략 어디쯤'일 뿐이라, 칸이 좁거나 캐릭터가 넓으면 슬롯 간격보다 몸이 넓어 겹친다.
    1) 아이 캐릭터: 슬롯 순서를 지키며, 높이가 겹치는 이웃과 겹치지 않을 만큼 좌우로 민다.
       칸 너비가 모자라면 모두 같은 비율로 조금씩 줄인다 (누구 하나만 작아지지 않게).
    2) 여러 번 나오는 조연: 아이 캐릭터가 차지하지 않은 가장 넓은 빈 구간 가운데에, 그 구간에 맞게 줄여 놓는다."""
    W, H = size
    gap = W * 0.02

    def box(c, s, sc, cx):
        w, h = s.width * sc, s.height * sc
        foot = min(c["foot"] * H, H - 2)
        return [cx - w / 2, foot - h, cx + w / 2, foot]

    def overlap_y(a, b):
        return min(a[3], b[3]) - max(a[1], b[1]) > 0.2 * min(a[3] - a[1], b[3] - b[1])

    kids = [i for i, c in enumerate(entries) if not c.get("extra")]
    kids.sort(key=lambda i: SLOTS[entries[i]["slot"]])
    base = {i: min(H * SIZES[entries[i]["size"]] / sprites[i].height, W * 0.45 / sprites[i].width) for i in kids}
    boxes = {}
    for shrink in (1.0, 0.9, 0.8, 0.7, 0.6):
        boxes = {i: box(entries[i], sprites[i], base[i] * shrink, SLOTS[entries[i]["slot"]] * W) for i in kids}
        for _ in range(30):                          # 겹친 이웃끼리 반씩 밀어내기
            moved = False
            for a, b in zip(kids, kids[1:]):
                A, B = boxes[a], boxes[b]
                push = A[2] + gap - B[0]
                if push > 0.5 and overlap_y(A, B):
                    for bx, d in ((A, -push / 2), (B, push / 2)):
                        bx[0] += d; bx[2] += d
                    moved = True
            for bx in boxes.values():               # 칸 밖으로 나가지 않게
                d = max(0 - bx[0], 0) - max(bx[2] - W, 0)
                bx[0] += d; bx[2] += d
            if not moved:
                break
        if all(boxes[a][2] + gap - boxes[b][0] <= 1 or not overlap_y(boxes[a], boxes[b]) for a, b in zip(kids, kids[1:])):
            break
    out = {i: (boxes[i][0], boxes[i][1], base[i] * shrink) for i in kids}
    taken = list(boxes.values())

    for i, c in enumerate(entries):
        if not c.get("extra"):
            continue
        s = sprites[i]
        sc = min(H * SIZES["small"] / s.height, W * 0.45 / s.width)
        probe = box(c, s, sc, W / 2)
        cuts = sorted((max(b[0] - gap, 0), min(b[2] + gap, W)) for b in taken if overlap_y(b, probe))
        free, x = [], 0.0
        for a, b in cuts:
            if a > x:
                free.append((x, a))
            x = max(x, b)
        free.append((x, W))
        lo, hi = max(free, key=lambda f: f[1] - f[0])
        # 빈 구간에 맞게 줄이되 절반 아래로는 줄이지 않는다 (점처럼 작아지면 알아볼 수 없다).
        # 그래도 모자라면 아이 캐릭터 뒤에 일부 가려진 채 놓인다 (작은 것부터 그리므로 뒤에 깔린다)
        sc = max(min(sc, (hi - lo) / s.width), sc * 0.5)
        b = box(c, s, sc, (lo + hi) / 2)
        out[i] = (b[0], b[1], sc)
        taken.append(b)
    return [out[i] for i in range(len(entries))]


def _place_layer(size, sprite, spot, flip):
    """페이지 크기 투명 층에 캐릭터를 spot=(x, y, 배율) 에 놓는다.
    sprite.info["points"](머리·손 좌표)도 같은 자리로 옮겨 돌려준다."""
    W, H = size
    s = ImageOps.mirror(sprite) if flip else sprite
    x, y, sc = spot
    s = s.resize((max(1, int(s.width * sc)), max(1, int(s.height * sc))), Image.LANCZOS)
    x = int(min(max(x, 0), W - s.width))
    y = int(max(y, 0))
    layer = Image.new("RGBA", size, (0, 0, 0, 0))
    layer.paste(s, (x, y), s)
    # 오려낸 가장자리가 칼로 자른 듯 딱 떨어지면 스티커처럼 보인다 -> 1픽셀 정도만 부드럽게 (더 흐리면 가는 펜 선이 흐려진다)
    layer.putalpha(layer.getchannel("A").filter(ImageFilter.GaussianBlur(1.2)))
    points = {}
    for name, (px, py) in sprite.info.get("points", {}).items():
        if flip:                                     # 좌우가 뒤집히면 왼손·오른손도 바뀐다
            px = sprite.width - px
            name = {"hand_left": "hand_right", "hand_right": "hand_left"}.get(name, name)
        points[name] = (x + px * sc, y + py * sc)
    return layer, points


def _shadow(page, layer, floating):
    box = layer.getchannel("A").getbbox()
    if not box:
        return page
    x0, y0, x1, y1 = box
    w = (x1 - x0) * (0.6 if floating else 1.1)
    cx = (x0 + x1) / 2
    cy = min(y1 + page.height * 0.08, page.height * 0.95) if floating else y1 + w * 0.03
    m = Image.new("L", page.size, 0)
    ImageDraw.Draw(m).ellipse((cx - w / 2, cy - w * 0.1, cx + w / 2, cy + w * 0.1),
                              fill=int(255 * (0.25 if floating else 0.55)))
    return Image.composite(Image.new("RGB", page.size, (60, 40, 30)), page,
                           m.filter(ImageFilter.GaussianBlur(w * 0.06 + 2)))


def _scene_light(bg, layer):
    """캐릭터에 '그 자리 주변'의 배경 색을 입히고(곱하기 25%), 가장자리에 배경의 밝은 빛을 둘러 같은 공간에 있는 느낌.
    그림 전체 평균색을 15% 섞던 방식은 흰색이 거의 그대로 남아, 흰 몸이 수채 배경 위에서 스티커처럼 떴다.
    곱하기라서 흰 곳일수록 주변 빛을 받고 진한 선·눈은 거의 그대로다. 25% 를 넘기면 초록 숲·밤 장면에서 캐릭터가 그 색으로 물든다."""
    alpha = layer.getchannel("A")
    rgb = np.array(layer.convert("RGB")).astype(np.float32)
    b = np.array(bg).astype(np.float32).reshape(-1, 3)
    around = np.array(bg.convert("RGB").filter(ImageFilter.GaussianBlur(bg.width // 16))).astype(np.float32)
    lit = rgb * 0.75 + rgb * (around / 255) * 0.25 * 1.15
    rim = np.array(alpha).astype(np.float32) - np.array(alpha.filter(ImageFilter.MinFilter(9))).astype(np.float32)
    r = np.array(Image.fromarray(np.clip(rim, 0, 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(3)))[..., None] / 255 * 0.35
    lit = lit * (1 - r) + np.percentile(b, 95, axis=0) * r
    out = Image.fromarray(np.clip(lit, 0, 255).astype(np.uint8)).convert("RGBA")
    out.putalpha(alpha)
    return out


def _gold_outline(page, shape):
    """아이 캐릭터만 두르는 금색 테두리. 굵은 띠(9px)와 넓은 번짐은 스티커 칼선처럼 보여, 얇은 띠(5px, 80%)와 은은한 번짐으로."""
    inside = shape.point(lambda v: 255 if v > 128 else 0)
    outside = np.array(ImageOps.invert(inside)) / 255
    glow = np.array(inside.filter(ImageFilter.MaxFilter(15)).filter(ImageFilter.GaussianBlur(8)))
    glow = Image.fromarray((glow * 0.45 * outside).astype(np.uint8))
    ring = Image.fromarray((np.array(inside.filter(ImageFilter.MaxFilter(5))) * outside * 0.8).astype(np.uint8))
    gold = Image.new("RGB", page.size, GOLD)
    page = Image.composite(gold, page, glow)
    return Image.composite(gold, page, ring.filter(ImageFilter.GaussianBlur(1.5)))


# 아이 캐릭터 발광 (2026-11-08 사용자: 배경에 녹아드는 건 좋은데 구분이 너무 약함 - gpt 그림처럼 빛나게).
# 그리는 것이 아니라 마지막에 빛을 합성하므로 모양은 바뀌지 않는다. 세기는 미리보기로 고른다.
# 첫 미리보기: 몸을 흰빛으로 밝히니 주황 몸이 바래 보였고, 밝은 해변에선 후광이 거의 안 보였다 ->
# 몸은 색을 유지한 채 밝게(곱하기), 윤곽 바로 바깥에 밝은 빛 띠(gpt 그림의 노랗게 빛나는 윤곽), 캐릭터 주변 배경을 더 어둡게.
GLOW_LEVELS = {
    "weak":   {"halo": 0.45, "edge": 0.45, "lift": 0.06, "dim": 0.12},
    "medium": {"halo": 0.65, "edge": 0.65, "lift": 0.10, "dim": 0.22},
    "strong": {"halo": 0.85, "edge": 0.85, "lift": 0.14, "dim": 0.32},
}
# 사용자 결정 (2026-11-08 미리보기 후): 강, 단 배경 어둡게는 절반
GLOW = dict(GLOW_LEVELS["strong"], dim=0.16)
HALO_COLOR, EDGE_COLOR = (255, 196, 80), (255, 236, 140)


def _screen(a, color, w):
    """밝게 더하기(스크린) 를 w(0~1, 픽셀마다) 만큼. 어두운 선은 거의 그대로, 밝은 곳은 빛 색으로."""
    lit = 255 - (255 - a) * (255 - np.array(color, np.float32)) / 255
    return a + (lit - a) * w[..., None]


def child_glow(page, shapes, halo, edge, lift, dim):
    """아이 캐릭터가 스스로 빛나는 느낌: ① 캐릭터 둘레 배경을 어둡게(dim, 캐릭터 바로 옆은 후광이 덮고 멀수록 어두움)
    ② 넓고 부드러운 후광(halo) ③ 윤곽 바로 바깥의 밝은 빛 띠(edge) ④ 몸을 색 그대로 한 단계 밝게(lift, 진한 선·눈 제외).
    빛의 크기는 캐릭터 크기에 비례한다 (작은 캐릭터에 큰 후광이 붙지 않게)."""
    masks = [np.array(s) > 128 for s in shapes]
    masks = [m for m in masks if m.any()]
    if not masks:
        return page
    a = np.array(page.convert("RGB")).astype(np.float32)
    union = np.any(masks, axis=0)
    size = np.mean([np.sqrt(m.sum()) for m in masks])
    far = 1 - np.exp(-ndimage.distance_transform_edt(~union) / (0.35 * size))
    a *= (1 - dim * far)[..., None]
    for m in masks:
        s = np.sqrt(m.sum())
        d = ndimage.distance_transform_edt(~m)
        a = _screen(a, HALO_COLOR, halo * np.exp(-d / (0.16 * s)) * ~m)
        a = _screen(a, EDGE_COLOR, edge * np.exp(-((d - 0.012 * s) / (0.012 * s)) ** 2) * ~m)
        lum = a.mean(axis=2)
        light = m & (lum > np.median(lum[m]) - 50)          # 진한 선·눈·입은 밝히지 않는다
        a = np.where(light[..., None], a * (1 + lift), a)    # 곱하기: 주황은 더 선명한 주황으로 (흰빛을 섞으면 바랜다)
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))


def _prop_layer(size, prop, owner_box, facing, points):
    """소품을 주인 캐릭터의 머리 위 / 손 / 옆 / 위 공중에 붙인다.
    머리 위·손은 바깥 상자가 아니라 자세 적용 뒤의 실제 좌표(points)를 쓴다 - 상자 꼭대기는 귀·뿔 끝일 수 있고,
    상자 옆면은 팔이 어디로 갔는지와 상관없기 때문이다. 팔이 없는 캐릭터는 바라보는 쪽 몸 옆에 붙인다."""
    x0, y0, x1, y1 = owner_box
    w, h = x1 - x0, y1 - y0
    anchor = prop["anchor"]
    rel_h = {"on_head": 0.32, "in_hand": 0.30, "beside": 0.40, "above": 0.30}[anchor]
    img = prop["image"]
    sc = h * rel_h / img.height
    img = img.resize((max(1, int(img.width * sc)), max(1, int(img.height * sc))), Image.LANCZOS)
    side = 1 if facing == "right" else -1
    hands = [p for n, p in points.items() if n.startswith("hand_")]
    if anchor == "on_head":
        hx, hy = points.get("head", ((x0 + x1) / 2, y0))
        x, y = hx - img.width / 2, hy - img.height * 0.85                  # 머리에 살짝 걸치게
    elif anchor == "in_hand" and hands:
        hx, hy = max(hands, key=lambda p: p[0] * side)                     # 바라보는 쪽 손
        x, y = hx - img.width / 2, hy - img.height / 2
    elif anchor == "in_hand":
        edge = x1 if side > 0 else x0
        x, y = edge - img.width / 2 + side * w * 0.05, y0 + h * 0.45 - img.height / 2
    elif anchor == "beside":
        x, y = (x1 + w * 0.05) if side > 0 else (x0 - w * 0.05 - img.width), y1 - img.height
    else:
        x, y = (x0 + x1) / 2 - img.width / 2, y0 - img.height * 1.15
    layer = Image.new("RGBA", size, (0, 0, 0, 0))
    layer.paste(img, (int(x), int(y)), img)
    return layer


def compose_panel(background, panel, assets, prop_images=None, extra_images=None, refine=None, glow=None):
    """background: 배경 그림. assets: {id: {"sprite", "rig", "child": True/False}}.
    prop_images: {소품 이름: RGBA}. extra_images: {조연 이름: {"sprite", "rig"}} (여러 번 나오는 조연).
    아이 그림 캐릭터에만 금색 테두리를 두르고, 조연·소품은 배경처럼 둔다.
    refine: refine(칸 그림, 붙인 것들의 자리 L, [아이 캐릭터 {"id", "layer": 붙인 층 RGBA}, ...])
    -> (칸 그림, 아이 캐릭터 모양 L 목록).
    금색 테두리를 두르기 전에 장면을 다듬는다 (테두리까지 다시 그리면 금색이 흐려진다). 아이 캐릭터를 장면 안에서
    다시 그리면 모양이 조금 바뀔 수 있으므로, 금색 테두리는 refine 이 돌려준 모양에 두른다."""
    page = background.convert("RGB")
    size = page.size
    placed = {}
    outlines = []
    child_layers = []
    pasted = Image.new("L", size, 0)               # 붙인 캐릭터·조연·소품 자리 (refine 이 안쪽 모양을 지키는 데 쓴다)

    entries = _entries(panel, assets, extra_images)
    posed_all = [pose(c["sprite"], c["rig"], c["pose"]) for c in entries]
    for c, posed in zip(entries, posed_all):
        c["foot"] = PLACES[c["place"]] - (0.08 if posed.info.get("float") and c["place"] == "ground" else 0)
    spots = _layout(size, entries, posed_all)
    for c, posed, spot in zip(entries, posed_all, spots):
        floating = c["place"] == "air" or posed.info.get("float")
        layer, points = _place_layer(size, posed, spot, flip=(c["facing"] == "left"))
        page = _shadow(page, layer, floating)
        layer = _scene_light(background.convert("RGB"), layer)
        page = Image.alpha_composite(page.convert("RGBA"), layer).convert("RGB")
        pasted = ImageChops.lighter(pasted, layer.getchannel("A"))
        placed[c["id"]] = (layer.getchannel("A").getbbox(), c["facing"], points)
        if c["child"]:
            outlines.append(layer.getchannel("A"))
            child_layers.append({"id": c["id"], "layer": layer})

    for pr in panel["props"]:
        if pr.get("painted") or not prop_images or pr["name"] not in prop_images or pr["owner"] not in placed:
            continue
        box, facing, points = placed[pr["owner"]]
        layer = _prop_layer(size, dict(pr, image=prop_images[pr["name"]]), box, facing, points)
        if pr["anchor"] == "beside":
            page = _shadow(page, layer, False)
        page = Image.alpha_composite(page.convert("RGBA"), _scene_light(background.convert("RGB"), layer)).convert("RGB")
        pasted = ImageChops.lighter(pasted, layer.getchannel("A"))

    if refine:
        page, outlines = refine(page, pasted, child_layers)
    if glow:                                        # glow: GLOW_LEVELS 의 값 (아이 캐릭터 발광)
        page = child_glow(page, outlines, **glow)
    for shape in outlines:                          # 금색 테두리는 마지막에 (소품에 가려지지 않게)
        page = _gold_outline(page, shape)
    return page


def _free_slots(panel):
    """여러 번 나오는 조연이 쓸 자리: 아이 캐릭터 자리에서 먼 빈 자리부터."""
    used = [SLOTS[c["slot"]] for c in panel["characters"]]
    return sorted((s for s in SLOTS if SLOTS[s] not in used),
                  key=lambda s: -min((abs(SLOTS[s] - u) for u in used), default=1))


def _entries(panel, assets, extra_images):
    """칸에 놓을 캐릭터 목록 (아이 캐릭터 + 여러 번 나오는 조연), 뒤(작은 것, 위쪽)부터 앞 순서.
    조연 자리는 _layout 이 아이 캐릭터를 놓고 남은 가장 넓은 빈 구간으로 정한다."""
    entries = [dict(c, child=assets[c["id"]].get("child", True), sprite=assets[c["id"]]["sprite"],
                    rig=assets[c["id"]]["rig"]) for c in panel["characters"]]
    for e in panel["extras"]:
        if e.get("recurring") and extra_images and e["name"] in extra_images:
            x = extra_images[e["name"]]
            entries.append({"id": e["name"], "extra": True, "size": "small", "place": "ground",
                            "pose": "stand", "facing": "left", "child": False, "sprite": x["sprite"], "rig": x["rig"]})

    # 뒤(작은 것, 위쪽)부터 앞으로 그린다
    entries.sort(key=lambda c: (PLACES[c["place"]], SIZES[c["size"]]))
    return entries


def reserved_mask(size, panel, assets, extra_images=None):
    """배경을 고르기 전에 쓰는 '캐릭터가 들어갈 자리' (L, 255 = 자리). 자세 전 모양으로 놓은 뒤 조금 넓힌다.
    배경 후보 중 이 자리에 눈에 띄는 물체가 없는 것을 고르는 데 쓴다."""
    m = np.zeros(size[::-1], np.uint8)
    # 아직 그리지 않은 여러 번 나오는 조연은 작은 네모로 자리만 잡는다 (배경을 조연보다 먼저 그리므로)
    extra_images = dict(extra_images or {})
    for e in panel["extras"]:
        if e.get("recurring") and e["name"] not in extra_images:
            extra_images[e["name"]] = {"sprite": Image.new("RGBA", (100, 100), (0, 0, 0, 255)), "rig": None}
    entries = _entries(panel, assets, extra_images)
    for c in entries:
        c["foot"] = PLACES[c["place"]]
    for c, spot in zip(entries, _layout(size, entries, [c["sprite"] for c in entries])):
        layer, _ = _place_layer(size, c["sprite"], spot, flip=(c["facing"] == "left"))
        m = np.maximum(m, np.array(layer.getchannel("A")))
    grown = ndimage.distance_transform_edt(m <= 128) <= size[1] * 0.02
    return Image.fromarray((grown * 255).astype(np.uint8))


PANEL_LAYOUTS = {
    1: [(0, 0, 1, 1)],
    2: [(0, 0, 1, 0.5), (0, 0.5, 1, 1)],
    3: [(0, 0, 1, 0.5), (0, 0.5, 0.5, 1), (0.5, 0.5, 1, 1)],
}


def panel_sizes(n_panels, page_size=(1024, 1024), border=8):
    """칸마다 배경을 따로 그릴 크기 (8의 배수)."""
    W, H = page_size
    out = []
    for x0, y0, x1, y1 in PANEL_LAYOUTS[n_panels]:
        w = int((x1 - x0) * W - border * 1.5) // 8 * 8
        h = int((y1 - y0) * H - border * 1.5) // 8 * 8
        out.append((w, h))
    return out


def assemble_page(panels, page_size=(1024, 1024), border=8):
    """칸 그림들을 흰 줄로 나눈 한 장으로 붙인다 (지금 앱의 single / two_panel / three_panel 레이아웃)."""
    W, H = page_size
    canvas = Image.new("RGB", page_size, "white")
    for img, (x0, y0, x1, y1) in zip(panels, PANEL_LAYOUTS[len(panels)]):
        if len(panels) == 1:
            canvas = img.resize(page_size, Image.LANCZOS)
            break
        bx, by = int(x0 * W + border * (0.5 if x0 else 1)), int(y0 * H + border * (0.5 if y0 else 1))
        canvas.paste(img, (bx, by))
    return canvas

In [ ]:
%%writefile story_pipeline.py
"""동화 그림 파이프라인 (API 없이 우리 모델로). Kaggle 시험 노트북(kaggle_full_story)과 그림 서버(kaggle_image_server)가 같이 쓴다.

    pipe = StoryPipeline().load()
    assets = pipe.make_characters({"hero": {"image": 아이그림(PIL), "desc": "영어 짧은 설명", "arms": 2, "legs": 0}})
    pictures = pipe.make_book(plan, assets)      # plan: scene_plan 형식 {"pages": [...]} -> 페이지 그림 목록

1. 아이 그림 -> 캐릭터 완성 (기본 SDXL 20단계 + scribble ControlNet + IP-Adapter), 오려내기·검사, 팔다리 후보
2. 칸마다 배경 먼저 (빠른 SDXL 8단계 1장, 캐릭터 자리가 막혔을 때만 다시 그림)
3. 들고 쓰는 소품·여러 번 나오는 조연: 처음 나오는 칸의 배경을 그림체 참고 그림으로 보여 주며 그림
4. 자세·그림자로 조립 -> 장면에 녹이기 (칸 전체를 약하게 다시 그림, 붙인 것 안쪽 모양은 유지) -> 금색 테두리 (puppet.py, scene_plan.py)
"""
import gc
import hashlib
import os
import time
import urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager

# GPU 메모리 조각화 방지 (torch 가 GPU 를 쓰기 전에 정해야 함)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import torch
from PIL import Image, ImageChops, ImageDraw, ImageEnhance, ImageFilter
from scipy import ndimage

from puppet import (ISNET_MIRROR, ISNET_SHA256, ISNET_URL, Matte, child_face, background_clutter, choose_limbs, cutout, cutout_problems, find_parts,
                    child_regions, keep_unpainted, numbered_overlay, outline_color, to_scribble)
from scene_plan import GLOW, assemble_page, background_prompt, compose_panel, normalize_plan, panel_sizes, reserved_mask

GOOD_STYLE = ("cozy storybook illustration, close view, warm golden afternoon light, soft glow, "
              "gentle pastel colors, gouache painting, clean simple shapes, whimsical, children's picture book")
# 따로 그리는 조연·소품은 배경과 같은 그림체 (장소·구도 말만 뺌). 아이 캐릭터만 아이 그림 모양을 따른다.
ASSET_STYLE = ("storybook illustration, warm soft light, gentle pastel colors, gouache painting, "
               "clean simple shapes, whimsical, children's picture book")
# 중요한 것부터 (StoryPipeline.negative 가 단계별 말을 앞에 붙이고 77토큰을 넘는 뒤쪽을 뺀다)
NEGATIVE = ("photo, realistic, 3d render, text, letters, watermark, signature, blurry, deformed, ugly, dark, scary, creepy, "
            "messy, muddy, oversaturated, neon colors, harsh contrast, cluttered, busy, too many details, "
            "wide landscape, panorama, horizon, dense foliage, pattern, wallpaper, tiled, caption, banner, title, logo")
NO_CHARACTERS = "people, person, human, child, animal, creature, character, face"
# 프롬프트에는 "무엇을 그릴지"만 쓴다. sticker·card·badge 같은 '포장' 말은 칼선 흰 테두리·바탕 카드까지 그리게 만든다.
CHARACTER_LOOK = "simple cute cartoon character, clean bold outline, flat soft colors, full body"
ALONE = "the whole thing fully visible, alone on a plain white background"
# 장면에 녹이기 지시문: 붙인 캐릭터가 그 장소의 빛·그림자 안에 있게
SCENE_TOUCH = "cute characters naturally part of the scene, soft shadows on the ground, warm light wrapping around them"
# 아이 캐릭터를 장면 안에서 다시 그릴 때: 모양은 선(ControlNet)이, 생김새는 완성 그림(IP-Adapter)이 지키고 글은 빛·질감만
LIVE_LOOK = ("storybook character standing in the scene, soft shading, warm light and gentle shadow, "
             "gouache texture, same character, children's picture book")
LIVE_IP = 0.6       # 완성 캐릭터 그림을 따르는 세기 (캐릭터 완성 때 0.5 보다 조금 세게 - 장면 색에 끌려가지 않게)
LIVE_SIZE = 768     # 캐릭터 둘레를 잘라 이 크기로 그린다
LIVE_STYLE_IP = 0.8     # 그림체 입히기: 그 칸 배경을 그림체 층에 보여 주는 세기
STYLE_REF_START = 0.5   # 조연·소품을 그릴 때 배경 그림체 참고 첫 세기 (걸리면 반으로, 마지막엔 0)
# 장면 안 다시 그리기 검사 기준 {점수: (최소, 최대)} - _live_scores 참고. 2026-11-08 처음 정한 값이라 비교 결과로 다시 맞춘다.
# (가짜 시험: 질감·음영·붓 자국은 통과, 몸이 배경으로 지워짐·얼굴 지움·새 만화 눈은 걸림)
LIVE_CHECK = {"body": (0.75, 9), "face": (0.6, 9), "new": (0, 0.3)}
# 그림체 입히기 때 얼굴을 새로 그리지 않게 (아이가 그린 얼굴은 선 지키기로 다시 덮는다)
LIVE_NO_FACE = "eyes, big eyes, mouth, smile, teeth, blush, rosy cheeks, nose, facial features"
BLOCKED = 0.25      # 캐릭터 자리의 이만큼 이상을 눈에 띄는 물체가 막으면 배경을 다시 그린다 (지난 시험 후보 대부분 0.05 이하)


def warm_grade(img, glow=True):
    """배경과 같은 따뜻한 색 보정. glow: 밝은 곳 번짐 - 오려 붙이는 그림에는 끈다 (가장자리에 밝은 띠가 생기므로)."""
    img = ImageEnhance.Color(img.convert("RGB")).enhance(0.9)
    a = np.array(img).astype(np.float32) * np.array([1.08, 1.02, 0.84])
    a = a * 0.96 + 255 * 0.04 * np.array([1.0, 0.85, 0.6])
    img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    if not glow:
        return img
    bright = img.point(lambda v: max(v - 170, 0) * 3).filter(ImageFilter.GaussianBlur(18))
    b, g = np.array(img).astype(np.float32), np.array(bright).astype(np.float32)
    return Image.fromarray(np.clip(255 - (255 - b) * (255 - g * 0.35) / 255, 0, 255).astype(np.uint8))


def graded(sprite):
    """오려낸 조연·소품에 배경과 같은 색 보정 (번짐 없이)."""
    out = warm_grade(sprite, glow=False).convert("RGBA")
    out.putalpha(sprite.getchannel("A"))
    return out


def gen_size(size, area=1024 * 1024):
    """칸 비율 그대로, SDXL 이 잘 그리는 넓이(약 100만 픽셀)로 키운 생성 크기 (64의 배수)."""
    w, h = size
    k = (area / (w * h)) ** 0.5
    return int(w * k) // 64 * 64, int(h * k) // 64 * 64


def style_only(strength):
    """IP-Adapter 를 '그림체 층'에만 건다 (InstantStyle): 참고 그림의 색·붓질만 가져오고 나무·길 같은 내용은 가져오지 않는다."""
    return {"up": {"block_0": [0.0, strength, 0.0]}}


def candidate_strip(images, blocked, reserved):
    """확인용: 그린 배경마다 캐릭터 자리(빨간 선)와 그 자리가 막힌 정도를 적은 줄 그림."""
    w = 384
    h = int(w * images[0].height / images[0].width)
    edge = reserved.resize((w, h)).filter(ImageFilter.FIND_EDGES).point(lambda v: 255 if v > 0 else 0)
    strip = Image.new("RGB", (w * len(images), h + 22), "white")
    for i, (img, busy) in enumerate(zip(images, blocked)):
        strip.paste(Image.composite(Image.new("RGB", (w, h), "red"), img.resize((w, h)), edge), (i * w, 0))
        ImageDraw.Draw(strip).text((i * w + 4, h + 4), f"blocked {busy:.2f}", fill="black")
    return strip


class StoryPipeline:
    def __init__(self, workdir="/kaggle/working", fast=True, character_steps=20, bg_tries=2, retries=2, seed=31,
                 save_dir=None, blend=0.0, inner_mix=0.3, live=0.6,
                 style=True, style_live=0.75, style_cn=0.7, line_keep=True, color_keep=True, live_fast=False,
                 live_enforce=True):
        """fast: 배경·조연·소품을 빠른 SDXL(Hyper-SD 8단계)로. 아이 캐릭터는 항상 기본 SDXL character_steps 단계
        (빠른 SDXL 은 ControlNet·IP-Adapter 와 함께 쓰면 색이 빠지고 선이 흐려진다).
        bg_tries: 배경은 1장 그리고, 캐릭터 자리가 막혔을 때만 다시 그려 최대 이 수만큼. retries: 검사에 걸렸을 때 다시 그리는 횟수.
        blend: 조립한 칸을 다시 그리는 세기 (0 = 장면에 녹이기 안 함 - 2026-11-08 비교에서 빼도 그림 차이가 거의 없어 기본 0, -33초). inner_mix: 붙인 것 안쪽에 다시 그린 그림을 섞는 비율
        (나머지는 원래 모양 그대로). save_dir: 중간 그림 저장 폴더 (확인용).
        live: 아이 캐릭터를 장면 안에서 다시 그리는 세기 (0 = 안 함, 붙인 그대로).
        style: 배경 그림체를 캐릭터에 입힌다 (세기 style_live, 선 따르기 style_cn). line_keep: 다시 그린 뒤 아이가 그린 선
        (윤곽·팔·눈·입)을 원래 그림에서 다시 덮는다. color_keep: 다시 그린 몸의 색조를 원래 색으로 되돌린다 (질감·명암은 둠).
        live_fast: 장면 안 다시 그리기도 빠른 SDXL(8단계)로.
        live_enforce: 검사(LIVE_CHECK)에 걸리면 버린다 (False = 점수만 기록)."""
        self.workdir, self.fast, self.character_steps = workdir, fast, character_steps
        self.bg_tries, self.retries, self.seed, self.save_dir = bg_tries, retries, seed, save_dir
        self.blend, self.inner_mix, self.live = blend, inner_mix, live
        self.style, self.style_live, self.style_cn = style, style_live, style_cn
        self.line_keep, self.color_keep = line_keep, color_keep
        self.live_fast, self.live_enforce = live_fast, live_enforce
        self.cpu = ThreadPoolExecutor(max_workers=1)   # 배경 제거 모델(CPU)을 GPU 작업과 겹쳐 돌린다
        self.glow = GLOW            # 아이 캐릭터 발광 (scene_plan.child_glow). None 이면 끔
        self.live_log = []          # 장면 안 다시 그리기 검사 점수 기록 (기준을 정할 때 본다)
        self.times = defaultdict(list)
        self.device, self.fp16 = "cuda", torch.float16
        self._characters = {}       # 같은 그림·설명이면 다시 그리지 않는다 (실패 페이지 다시 그리기에서도 같은 모습 유지)
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)

    # ── 준비 ──────────────────────────────────────────────────────
    @contextmanager
    def timed(self, label):
        torch.cuda.synchronize()
        t = time.perf_counter()
        yield
        torch.cuda.synchronize()
        self.times[label].append(time.perf_counter() - t)

    def _save(self, img, name):
        if self.save_dir:
            img.save(os.path.join(self.save_dir, name))

    def load(self):
        from diffusers import (AutoencoderKL, ControlNetModel, DDIMScheduler, DPMSolverMultistepScheduler,
                               StableDiffusionXLControlNetInpaintPipeline, StableDiffusionXLControlNetPipeline,
                               StableDiffusionXLImg2ImgPipeline, StableDiffusionXLPipeline)

        with self.timed("모델 불러오기 (처음 1번)"):
            vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=self.fp16)
            self.sdxl = StableDiffusionXLPipeline.from_pretrained(
                "stabilityai/stable-diffusion-xl-base-1.0", vae=vae, torch_dtype=self.fp16, variant="fp16",
                use_safetensors=True).to(self.device)
            self.base_sched = DPMSolverMultistepScheduler.from_config(self.sdxl.scheduler.config, use_karras_sigmas=True)
            self.fast_sched = DDIMScheduler.from_config(self.sdxl.scheduler.config, timestep_spacing="trailing")  # Hyper-SD 권장
            self.sdxl.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models",
                                      weight_name="ip-adapter-plus_sdxl_vit-h.safetensors",
                                      image_encoder_folder="models/image_encoder")
            # 빠른 SDXL: 기본 모델에 8단계용 LoRA 를 얹는다 (CFG 버전 = negative prompt 를 그대로 쓸 수 있음). 켜고 끌 수 있다.
            self.sdxl.load_lora_weights("ByteDance/Hyper-SD", weight_name="Hyper-SDXL-8steps-CFG-lora.safetensors",
                                        adapter_name="hyper")
            # ControlNet 은 아이 캐릭터를 그릴 때만 GPU 에 올린다 (장면 안에서 다시 그리기(live)를 끄면 다 그린 뒤 CPU 로 내림)
            self.controlnet = ControlNetModel.from_pretrained("xinsir/controlnet-scribble-sdxl-1.0", torch_dtype=self.fp16)
            self.sheet_pipe = StableDiffusionXLControlNetPipeline(**self.sdxl.components, controlnet=self.controlnet)
            # 장면에 녹이기: 조립한 칸을 약하게 다시 그린다 (같은 모델을 나눠 쓰므로 메모리가 더 들지 않는다)
            self.refine_pipe = StableDiffusionXLImg2ImgPipeline(**self.sdxl.components)
            # 아이 캐릭터를 장면 안에서 다시 그리기: 붙인 캐릭터 자리만, 자세 그대로의 선(scribble)과 캐릭터 그림(IP-Adapter)을 보며
            self.live_pipe = StableDiffusionXLControlNetInpaintPipeline(**self.sdxl.components, controlnet=self.controlnet)
            # 학습된 배경 제거 모델 (CPU). 오려내기, 배경에서 캐릭터 자리를 막는 물체 찾기에 쓴다
            path = os.path.join(self.workdir, "isnet-general-use.onnx")
            fetch_isnet(path)
            self.matte = Matte(path)
        for p in (self.sdxl, self.sheet_pipe, self.refine_pipe, self.live_pipe):
            p.set_progress_bar_config(disable=True)
        # 그림으로 바꾸는 마지막 단계(VAE 디코딩)를 조각내서 한다: 필요한 메모리가 그림 해상도와 상관없이 일정해진다
        self.sdxl.vae.enable_tiling()   # 파이프라인의 enable_vae_tiling() 은 최신 diffusers 에서 없어짐 - VAE 에 직접 건다
        self.white = Image.new("RGB", (224, 224), "white")   # IP-Adapter 를 안 쓸 때 넣는 빈 그림 (scale 0 과 함께)
        self.set_speed(self.fast)
        self.gpu_report("모델을 모두 올린 뒤")
        return self

    def gpu_report(self, label):
        print(f"[GPU] {label}: 사용 {torch.cuda.memory_allocated() / 1e9:.1f} GB, "
              f"여유 {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB", flush=True)

    def gen(self, seed):
        return torch.Generator(self.device).manual_seed(seed)

    def set_speed(self, fast, steps=None):
        """빠른 SDXL(Hyper-SD LoRA + DDIM trailing, 8단계) <-> 기본(DPM++ Karras). 두 파이프라인이 같은 UNet 을 쓴다."""
        for p in (self.sdxl, self.sheet_pipe, self.refine_pipe, self.live_pipe):
            p.scheduler = self.fast_sched if fast else self.base_sched
        (self.sdxl.enable_lora if fast else self.sdxl.disable_lora)()
        self.steps = steps or (8 if fast else 30)

    def fit_prompt(self, content, tail):
        """SDXL 은 지시문 앞쪽 77토큰만 읽는다. 장면 설명(content)이 길면 뒤에 붙인 그림체·'empty scene' 같은 말(tail)이
        잘려 나가므로, 넘칠 때는 tail 은 그대로 두고 content 를 뒤에서부터 단어 단위로 줄인다."""
        tok = getattr(self.sdxl, "tokenizer", None)
        words = content.split()
        while tok is not None and words:
            prompt = f"{' '.join(words)}, {tail}"
            if len(tok(prompt).input_ids) <= tok.model_max_length:
                return prompt
            words.pop()
        return f"{' '.join(words)}, {tail}" if words else tail

    def negative(self, *specific):
        """단계별 금지어(specific)를 앞에 두고 공통 금지어(NEGATIVE)를 77토큰이 찰 때까지 붙인다.
        SDXL 은 negative 도 앞 77토큰만 읽는데 NEGATIVE 만으로 약 75토큰이라, 예전처럼 NEGATIVE 뒤에 붙인 말
        (배경의 '사람·동물 빼기', 캐릭터·소품의 '배경 풍경 빼기', 다시 그리기의 '얼굴 그리지 마')은 모두 잘려 효과가 없었다
        (2026-11-08 발견 - 냄비가 매번 풍경 섞임으로 3번 그려진 것과 관련 추정)."""
        items = [x.strip() for part in specific for x in part.split(",") if x.strip()]
        common = [x.strip() for x in NEGATIVE.split(",")]
        tok = getattr(self.sdxl, "tokenizer", None)
        if tok is None:
            return ", ".join(items + common)
        out = list(items)
        for x in common:
            if len(tok(", ".join(out + [x])).input_ids) > tok.model_max_length:
                break
            out.append(x)
        return ", ".join(out)

    def txt2img(self, prompt, negative, size, seed):
        self.sdxl.set_ip_adapter_scale(0.0)
        return self.sdxl(prompt, negative_prompt=negative, width=size[0], height=size[1], guidance_scale=5.5,
                         num_inference_steps=self.steps, generator=self.gen(seed), ip_adapter_image=self.white).images[0]

    # ── 검사·점수 ────────────────────────────────────────────────
    def drawn_sprite(self, make, label, seed, inside=None):
        """그리기 -> 학습된 모델로 오려내기 -> 검사: 포장(바탕 카드·흰 테두리 띠), 바탕에 풍경이 따라 그려짐(busy).
        걸리면 seed 를 바꿔 최대 retries 번 다시 그린다 (make(seed, 몇 번째, 마지막인가) - 그림체 참고 세기도 줄인다).
        끝까지 걸리면 문제가 가장 적고, 같으면 풍경이 가장 적게 섞인 것을 쓴다
        ('같으면 첫 번째'로 고르면 가장 세게 참고한 첫 그림이 다시 뽑혀 다시 그린 보람이 없다).
        inside: 아이 그림 기준 몸 안쪽 (아이 캐릭터만) - 생성 그림의 선이 끊겨도 안쪽이 투명하게 잘리지 않게 오려내기에 더한다."""
        best = None
        for t in range(self.retries + 1):
            img = make(seed + 1000 * t, t, t == self.retries)
            mt = self.matte(img)
            sprite = cutout(img, matte=mt, inside=inside)
            clutter = background_clutter(img, mt)
            # 물체 바깥의 1/3 이상이 바탕색과 뚜렷이 다르면 '혼자 흰 바탕'이 아니라 풍경까지 그려진 것
            problems = cutout_problems(sprite) + (["busy"] if clutter > 1 / 3 else [])
            rank = (len(problems), clutter)
            if best is None or rank < best[2]:
                best = (img, sprite, rank)
            if not problems:
                break
            self._save(img, f"try_{_safe(label)}_{t}.png")          # 다시 그린 까닭을 나중에 볼 수 있게
            print(f"   {label}: 오려내기 검사 {problems} (풍경 섞임 {clutter:.2f})"
                  + (" -> 다시 그리기" if t < self.retries else " -> 가장 나은 것 사용"), flush=True)
        return best[0], best[1]

    @staticmethod
    def blocked(matte, reserved):
        """캐릭터 자리(reserved_mask)를 눈에 띄는 물체(배경 제거 모델 출력)가 막고 있는 정도 0~1."""
        spot = np.array(reserved.convert("L")) > 128
        return float((np.array(matte.resize(reserved.size)) / 255.0)[spot].mean()) if spot.any() else 0.0

    def refine(self, page, pasted, kids, text, seed):
        """장면에 녹이기: 조립한 칸 전체를 빠른 SDXL 로 약하게(blend) 다시 그린다 -> 붙인 가장자리가 배경에 녹고
        바닥 그림자·빛·종이 질감이 캐릭터까지 이어진다. 다시 그리면 아이 그림 모양이 흔들릴 수 있으므로
        조연·소품(pasted)은 다시 그린 그림을 inner_mix 만큼만 섞는다 (2px 안으로 줄인 자리).
        아이 캐릭터(kids)는 다시 그린 그림을 섞지 않고, 자리를 3px 넓혀 원래 그대로 둔다: 2026-11-08 Kaggle 비교에서
        섞으면 세기 0.3 부터 풀밭 위 가는 팔이 나뭇가지로 바뀌거나 사라지고, 0.4 부터 하얀 얼굴에 부리·볼터치가 새로 생겼다
        (반투명한 가는 선은 자리 기준(128)에도 못 들어 32 로 낮춤). 아이 캐릭터는 이미 _scene_light 로 주변 빛을 받는다.
        붙인 것 바깥(그림자·주변 배경)은 다시 그린 그림을 그대로 써서 캐릭터와 배경 사이 경계가 이어진다."""
        if self.blend <= 0:
            return page
        size = page.size
        self.sdxl.set_ip_adapter_scale(0.0)
        with self.timed("장면에 녹이기 (칸 1개)"):
            out = self.refine_pipe(self.fit_prompt(text, f"{SCENE_TOUCH}, {GOOD_STYLE}"), negative_prompt=NEGATIVE,
                                   image=page.resize(gen_size(size), Image.LANCZOS), strength=self.blend,
                                   guidance_scale=5.5, num_inference_steps=self.steps, generator=self.gen(seed),
                                   ip_adapter_image=self.white).images[0].resize(size, Image.LANCZOS)
        inner = pasted.point(lambda v: 255 if v > 128 else 0).filter(ImageFilter.MinFilter(5))
        keep = np.array(inner.filter(ImageFilter.GaussianBlur(2))).astype(np.float32)[..., None] / 255 * (1 - self.inner_mix)
        child = kids.point(lambda v: 255 if v > 32 else 0).filter(ImageFilter.MaxFilter(7)).filter(ImageFilter.GaussianBlur(1.5))
        keep = np.maximum(keep, np.array(child).astype(np.float32)[..., None] / 255)
        mixed = np.array(out).astype(np.float32) * (1 - keep) + np.array(page).astype(np.float32) * keep
        return Image.fromarray(np.clip(mixed, 0, 255).astype(np.uint8))

    def liven(self, page, item, pasted, asset, text, seed, name, background):
        """아이 캐릭터를 장면 안에서 다시 그린다 (붙인 스티커 -> 그 장소에 있는 캐릭터).
        - 어디를: 붙인 캐릭터 자리 + 둘레 띠(자른 그림의 4%)만. 둘레까지 다시 그려야 가장자리·바닥이 배경과 이어진다.
          다른 캐릭터·소품 자리는 빼서 지운다 (손에 든 냄비가 지워지지 않게).
        - 무엇을 따라: 자세를 적용한 캐릭터의 선(scribble ControlNet) = 모양, 지금 칸 그림에서 출발 = 색 배치,
          IP-Adapter = 완성 캐릭터 그림(기본) 또는 그 칸 배경의 그림체(style).
        - 얼마나 크게: 캐릭터 둘레를 정사각형으로 잘라 LIVE_SIZE 로 키워 그린다 (작은 캐릭터도 선이 안 뭉개지고, 칸 전체보다 빠르다).
        - 그림체 입히기(style): 배경을 IP-Adapter 그림체 층에만 보여 주고(LIVE_STYLE_IP), 선을 style_cn 으로, 세기 style_live.
          2026-11-08 Kaggle: 수채·동화풍으로 크게 바뀌어 gpt 그림에 가장 가까웠지만, 아이가 그린 점·일자 눈이 큰 만화 눈·볼터치·
          부리·이빨로 바뀌었다 -> 얼굴 그리지 말라는 지시(LIVE_NO_FACE)를 negative 에 넣고,
          line_keep: 다시 그린 뒤 아이가 그린 선 전부(윤곽선·막대 팔·눈·입 = 몸 색보다 뚜렷이 진한 곳)를 원래 그림에서 다시 덮는다
          -> 선은 아이 그림 그대로, 안쪽 색·질감만 그림체. (얼굴만 덮던 방식은 막대 팔이 흐려지거나 사라졌다.)
        - (행동 모드는 없앴다: SDXL 이 행동을 그리지 못하고 지운 팔 자리가 흰 안개로 덮였으며, 팔다리 자리 넓히기가 1번에 20초 걸렸다.
          행동은 꼭두각시 자세(puppet.pose)로 표현한다.)
        - 검사 (_live_scores): 몸이 남았나(배경과 다른 색으로 남았나) / 원래 얼굴 표시가 남았나 / 몸 안에 새 얼굴·무늬가 생겼나.
          기준 LIVE_CHECK. live_enforce=False 면 버리지 않고 점수만 남긴다 (기준을 정하는 비교용).
          (배경 제거 모델로 모양을 재던 방식은 밝은 배경 위 하얀·노란 캐릭터를 놓쳐 좋은 그림을 버렸다 - 2026-11-08)
        -> (칸 그림, 금색 테두리를 두를 모양 L = 원래 붙인 모양. 선을 원래대로 덮으므로 모양이 같다)."""
        layer = item["layer"]
        alpha = layer.getchannel("A")
        if self.live <= 0 or not alpha.getbbox():
            return page, alpha
        W, H = page.size
        x0, y0, x1, y1 = alpha.getbbox()
        side = int(min(max(x1 - x0, y1 - y0) * 1.3, W, H))
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        bx = int(min(max(cx - side / 2, 0), W - side))
        by = int(min(max(cy - side / 2, 0), H - side))
        box = (bx, by, bx + side, by + side)
        S = LIVE_SIZE

        def cut(img, resample=Image.LANCZOS):
            return img.crop(box).resize((S, S), resample)

        shape = np.array(cut(alpha)) > 32
        # 다른 캐릭터·소품 자리 (붙인 것 전체 - 이 캐릭터), 조금 넓혀서
        others = _grow((np.array(cut(pasted)) > 64) & ~shape, 2)
        mask = Image.fromarray(((_grow(shape, S * 0.02) & ~others) * 255).astype(np.uint8))
        core = ndimage.distance_transform_edt(shape) > 3
        ring = max(3, int(S * 0.04))
        lines = Image.new("RGB", layer.size, "white")
        lines.paste(layer, (0, 0), layer)
        crop = cut(page)
        control = to_scribble(cut(lines), size=S)
        prompt = self.fit_prompt(f"{asset['look']}, in {text}", LIVE_LOOK)
        self.set_speed(self.live_fast, None if self.live_fast else self.character_steps)
        if self.style:     # 그림체 층에만 그 칸 배경을 참고 그림으로
            self.sdxl.set_ip_adapter_scale(style_only(LIVE_STYLE_IP))
            ref, cn, strength = background.convert("RGB"), self.style_cn, self.style_live
        else:              # 완성 캐릭터 그림 전체를 참고 그림으로
            self.sdxl.set_ip_adapter_scale(LIVE_IP)
            ref, cn, strength = asset["sheet"], 0.8, self.live
        with self.timed("아이 캐릭터 장면 안에서 다시 그리기 (1번)"):
            out = self.live_pipe(prompt, negative_prompt=self.negative(LIVE_NO_FACE) if self.style else self.negative(),
                                 image=crop, mask_image=mask, control_image=control,
                                 width=S, height=S, strength=strength, controlnet_conditioning_scale=cn,
                                 guidance_scale=6.0, num_inference_steps=self.steps, generator=self.gen(seed),
                                 ip_adapter_image=ref).images[0]
        self.set_speed(self.fast)
        drawn = out
        if self.color_keep:  # 몸 색조를 원래 색으로 (그림체를 따라 주황이 노랑으로 바뀌던 문제), 그다음 부분 얼룩(볼터치)도 되돌림
            body_px = core & ~_child_lines(crop, shape, core)
            out = _drop_new_colors(crop, _keep_colors(crop, out, body_px), body_px)
        if self.line_keep:   # 아이가 그린 선(윤곽·팔·눈·입)을 원래 그림에서 다시 덮는다
            ink = _child_lines(crop, shape, core)
            if ink.any():
                soft = Image.fromarray((_grow(ink, 1.5) * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(0.8))
                out = Image.composite(crop, out, soft)
        with self.timed("다시 그린 모양 검사 (CPU)"):
            bg_crop = cut(background.convert("RGB"))
            sc = _live_scores(crop, out, core, bg_crop)
            # 몸이 남았나는 모델이 그린 그대로(색·선 되돌리기 전)로 본다: 몸이 배경으로 지워졌어도 색 지키기가
            # 그 자리를 캐릭터 색으로 다시 물들이면 '남은 것'처럼 보인다 (노트북 가짜 시험에서 발견)
            sc["body"] = _live_scores(crop, drawn, core, bg_crop)["body"]
            fails = [k for k, (lo, hi) in LIVE_CHECK.items() if not (lo <= sc[k] <= hi)]
            ok = not fails
        mode = (("그림체" + (" + 선 지키기" if self.line_keep else "") + (" + 색 지키기" if self.color_keep else ""))
                if self.style else "자세 그대로")
        score_en = " ".join(f"{k} {v:.3f}" if k == "new" else f"{k} {v:.2f}" for k, v in sc.items())
        verdict = "OK" if ok else f"FAIL({','.join(fails)})"
        print(f"   {name}: 장면 안 다시 그리기 ({mode}) {score_en} -> "
              f"{'사용' if ok else ('기준 미달이지만 사용 (비교용)' if not self.live_enforce else '버림 (붙인 그대로)')}"
              + ("" if ok else f" - 걸린 것 {fails}"), flush=True)
        self.live_log.append({"name": name, "mode": mode, **sc, "ok": ok})
        if self.save_dir:
            shots = (crop, control, drawn) + ((out,) if out is not drawn else ())
            strip = Image.new("RGB", (S * len(shots), S + 24), "white")
            for k, im in enumerate(shots):
                strip.paste(im, (k * S, 24))
            ImageDraw.Draw(strip).text((6, 6), "before | lines | after" + (" | lines kept" if len(shots) == 4 else "")
                                       + f"   {'STYLE' if self.style else 'POSE'}   {score_en}  {verdict}", fill="black")
            strip.resize((strip.width // 2, strip.height // 2)).save(os.path.join(self.save_dir, f"live_{_safe(name)}.png"))
        if not ok and self.live_enforce:
            return page, alpha
        # 다시 그린 자리만 부드럽게 바꿔 넣는다 (자르고 키운 그림을 원래 크기로)
        soft = mask.filter(ImageFilter.GaussianBlur(ring / 4))
        page = page.copy()
        page.paste(out.resize((side, side), Image.LANCZOS), box[:2], soft.resize((side, side), Image.LANCZOS))
        return page, alpha

    # ── 1. 아이 캐릭터 ────────────────────────────────────────────
    def make_characters(self, characters, compare=False):
        """characters: {id: {"image": 아이 그림(PIL), "desc": 영어 짧은 설명, "arms": 개수, "legs": 개수}}
        -> {id: {"sprite", "rig", "child": True}}. compare: 기본 30단계로도 그려 나란히 저장 (확인용)."""
        todo = {k: v for k, v in characters.items() if self._key(v) not in self._characters}
        if todo:
            self.controlnet.to(self.device)
            self.sdxl.set_ip_adapter_scale(0.5)
            self.set_speed(False, self.character_steps)
            for i, (k, info) in enumerate(todo.items()):
                drawing = _on_white(info["image"])
                self._save(to_scribble(drawing), f"scribble_{k}.png")
                inside, unpainted = child_regions(drawing, 1024)
                make, measured = self._sheet_maker(info, drawing, unpainted, inside)
                with self.timed(f"캐릭터 완성 (1명, 기본 {self.character_steps}단계, 검사·다시 그리기 포함)"):
                    sheet, sprite = self.drawn_sprite(make, k, self.seed + i, inside=inside)
                self._save(sheet, f"sheet_{k}.png")
                if compare:
                    self.set_speed(False, 30)
                    with self.timed("캐릭터 완성 (기본 30단계, 비교용)"):
                        base = make(self.seed + i, 0)
                    self.set_speed(False, self.character_steps)
                    pair = Image.new("RGB", (2048, 1024), "white")
                    pair.paste(sheet, (0, 0)); pair.paste(base, (1024, 0))
                    self._save(pair.resize((1024, 512)), f"compare_sheet_{k}.png")   # 왼쪽 20단계 | 오른쪽 30단계
                rig = choose_limbs(find_parts(sprite), arms=info.get("arms"), legs=info.get("legs"))
                self._save(numbered_overlay(sprite, rig), f"limbs_{k}.png")
                # sheet·look: 장면 안에서 다시 그릴 때 IP-Adapter 참고 그림과 지시문으로 쓴다
                self._characters[self._key(info)] = {"sprite": sprite, "rig": rig, "child": True, "sheet": sheet,
                                                     "look": ", ".join([info["desc"]] + measured)}
                print(f"{k}: 후보 {len(rig.limbs)}개, 움직이는 것 {[f'{l.id}:{l.kind}' for l in rig.moving]}", flush=True)
            # 장면 안에서 다시 그리기를 안 쓰면 ControlNet 은 CPU 로 내린다 (다음 동화에서 다시 올림).
            # IP-Adapter 는 조연·소품 그림체 참고에 쓰므로 둔다.
            if self.live <= 0:
                self.controlnet.to("cpu")
                gc.collect(); torch.cuda.empty_cache()
            self.set_speed(self.fast)
        return {k: self._characters[self._key(v)] for k, v in characters.items()}

    def _sheet_maker(self, info, drawing, unpainted, inside):
        # 아이 그림에서 잰 것을 글로도 알려 준다 (지시문이 길어도 잘리지 않게 뒤쪽 고정 부분에 넣는다):
        # 윤곽선 색 - 주황 몸의 갈색 테두리처럼 면에 붙은 선은 그림만으로는 사라지기 쉽다
        # white inside - 안쪽을 칠하지 않았는데 선 색이 글에 있으면 안쪽까지 그 색으로 칠하기 쉽다
        measured = []
        line = outline_color(drawing)
        if line:
            measured.append(f"{line} outline")
        if inside.any() and unpainted.sum() > 0.5 * inside.sum():
            measured.append("white inside")
        prompt = self.fit_prompt(info["desc"], ", ".join(measured + [CHARACTER_LOOK, "plain white background"]))

        def make(seed, t=0, last=False):
            sheet = self.sheet_pipe(prompt, negative_prompt=self.negative("background scenery"),
                                    image=to_scribble(drawing), ip_adapter_image=drawing,
                                    controlnet_conditioning_scale=0.9, width=1024, height=1024,
                                    guidance_scale=6.0, num_inference_steps=self.steps, generator=self.gen(seed)).images[0]
            # 그래도 아이가 비워 둔 곳에 색이 생기면 그린 뒤 하얗게 되돌린다 (다시 그리기보다 빠르다).
            # 몸 안 표시(눈·코·입)는 아이 그림의 것으로 바꾼다 - 완성 단계가 코·웃는 입·안경을 지어내던 문제 (2026-11-11)
            return child_face(keep_unpainted(sheet, unpainted), drawing, inside)
        return make, measured

    @staticmethod
    def _key(info):
        h = hashlib.sha1(info["image"].convert("RGBA").tobytes())
        h.update(f"{info.get('desc')}|{info.get('arms')}|{info.get('legs')}".encode())
        return h.hexdigest()

    # ── 2~4. 배경 -> 조연·소품 -> 조립 ───────────────────────────────
    def make_book(self, plan, assets):
        """plan: scene_plan 형식 {"pages": [...]}. assets: make_characters 결과. -> 페이지 그림(PIL) 목록."""
        pages = normalize_plan(plan, list(assets))
        backgrounds, style_ref = self._backgrounds(pages, assets)
        props, extras = self._side_pieces(pages, style_ref)
        self.last_book = (pages, backgrounds, assets, props, extras)    # 시험 노트북이 녹이기 세기만 바꿔 다시 조립할 때 쓴다
        return self.compose_pages(pages, backgrounds, assets, props, extras)

    def compose_pages(self, pages, backgrounds, assets, props, extras, tag=""):
        """칸 조립 -> 아이 캐릭터 장면 안에서 다시 그리기 -> 장면에 녹이기 -> 금색 테두리 -> 페이지.
        배경·조연·소품을 다시 그리지 않고 세기만 바꿔 비교할 때도 쓴다."""
        pictures = []
        for n, page in enumerate(pages):
            panels = []
            for i, panel in enumerate(page["panels"]):
                text = background_prompt(panel)
                seed = self.seed + 900 + 100 * (n + 1) + 10 * i

                def refine(img, pasted, child_layers, text=text, seed=seed, where=f"p{page['page']}_{i + 1}",
                           bg=backgrounds[(n, i)]):
                    shapes = []
                    for j, item in enumerate(child_layers):
                        img, shape = self.liven(img, item, pasted, assets[item["id"]], text, seed + j,
                                                f"{item['id']}_{where}{tag}", bg)
                        shapes.append(shape)
                    kids = Image.new("L", img.size, 0)
                    for shape in shapes:
                        kids = ImageChops.lighter(kids, shape)
                    return self.refine(img, pasted, kids, text, seed), shapes
                with self.timed("칸 조립 (자세·그림자·테두리, 녹이기 포함)"):
                    panels.append(compose_panel(backgrounds[(n, i)], panel, assets, props, extras, refine=refine,
                                                glow=self.glow))
            picture = assemble_page(panels)
            self._save(picture, f"page{page['page']}{tag}.png")
            pictures.append(picture)
        return pictures

    def _backgrounds(self, pages, assets):
        """칸마다 배경 먼저: 1장 그리고, 캐릭터 자리를 눈에 띄는 물체가 막았을 때만 다시 그린다 (최대 bg_tries 장).
        예전처럼 3장을 그려 CLIP 으로 고르던 방식은 후보 차이가 작아도 1등·꼴찌를 억지로 벌려 사실상 무작위였고,
        자리 점수도 거의 모든 후보가 0.95 이상이라 시간만 썼다.
        한 번만 나오는 조연·옆에 놓이기만 하는 소품은 배경과 함께 그린다 (normalize_plan 이 painted 표시).
        여러 번 나오는 조연은 아직 그리기 전이라 reserved_mask 가 작은 네모 크기로 자리만 비워 둔다."""
        backgrounds, style_ref = {}, {}
        jobs = []          # (n, i, panel, size, text, negative, reserved, 첫 장, 자리 검사 작업)
        for n, page in enumerate(pages):
            for i, (panel, size) in enumerate(zip(page["panels"], panel_sizes(len(page["panels"])))):
                text = background_prompt(panel)
                once = [e for e in panel["extras"] if not e.get("recurring")]
                negative = self.negative('people, person, human, child, face' if once else NO_CHARACTERS)  # 배경 속 조연 허용
                reserved = reserved_mask(gen_size(size), panel, assets)
                with self.timed("배경 (1장)"):
                    first = self.txt2img(self.fit_prompt(text, f"{GOOD_STYLE}, empty scene"), negative, gen_size(size),
                                         self.seed + 100 * (n + 1) + 10 * i)
                # 자리 검사(배경 제거 모델, CPU)는 GPU 가 다음 칸을 그리는 동안 따로 돌린다
                jobs.append((n, i, panel, size, text, negative, reserved, first, self.cpu.submit(self.matte, first)))
        for n, i, panel, size, text, negative, reserved, first, job in jobs:
            page = pages[n]
            cands = [first]
            with self.timed("배경 자리 검사 (CPU, 기다린 시간)"):
                blocked = [self.blocked(job.result(), reserved)]
            for c in range(1, self.bg_tries):
                if blocked[-1] <= BLOCKED:
                    break
                with self.timed("배경 (1장)"):
                    cands.append(self.txt2img(self.fit_prompt(text, f"{GOOD_STYLE}, empty scene"), negative, gen_size(size),
                                              self.seed + 100 * (n + 1) + 10 * i + c))
                with self.timed("배경 자리 검사 (CPU, 기다린 시간)"):
                    blocked.append(self.blocked(self.matte(cands[-1]), reserved))
            name = f"p{page['page']}_{i + 1}"
            if self.save_dir:
                candidate_strip(cands, blocked, reserved).save(os.path.join(self.save_dir, f"cands_{name}.png"))
            best = int(np.argmin(blocked))
            bg = warm_grade(cands[best]).resize(size, Image.LANCZOS)
            self._save(bg, f"bg_{name}.png")
            backgrounds[(n, i)] = bg
            for pr in panel["props"]:
                if not pr["painted"]:
                    style_ref.setdefault(pr["name"], bg)
            for e in panel["extras"]:
                if e.get("recurring"):
                    style_ref.setdefault(e["name"], bg)
            print(f"{page['page']}페이지 {i + 1}칸: {text}\n   캐릭터 자리 막힘 {[round(b, 2) for b in blocked]} -> {best + 1}번",
                  flush=True)
        return backgrounds, style_ref

    def _styled(self, ref, prompt, negative):
        """그림체 참고 세기 STYLE_REF_START 로 그리고, 검사(풍경이 따라 그려짐 등)에 걸려 다시 그릴 때마다 반으로 줄인다.
        (1.0 에서 시작하면 참고 그림(배경)의 풍경까지 따라와 자주 걸렸다 - 2026-11-08 강아지 3번 모두 busy, 42초.)
        마지막 다시 그리기는 참고를 끈다(0) - 어떤 물체든 마지막엔 흰 바탕에 혼자 있는 그림이 보장되고,
        그림체는 글자 지시(ASSET_STYLE)와 색 보정(graded)으로 맞춘다.
        물체마다 참고 그림에 끌려가는 정도가 달라서 (강아지는 괜찮고 냄비는 풍경에 묻힘) 하나로 정한 세기는 맞지 않는다."""
        def make(seed, t, last=False):
            self.sdxl.set_ip_adapter_scale(style_only(0.0 if last and t > 0 else STYLE_REF_START * 0.5 ** t))
            return self.sdxl(prompt, negative_prompt=negative, width=1024, height=1024, guidance_scale=5.5,
                             num_inference_steps=self.steps, generator=self.gen(seed), ip_adapter_image=ref).images[0]
        return make

    def _side_pieces(self, pages, style_ref):
        """들고 쓰는 소품·여러 번 나오는 조연: 처음 나오는 칸의 배경을 그림체 참고 그림으로 보여 주며 그린다.
        그림체를 글자로만 지시하면 흰 바탕에 혼자 그린 동물은 SDXL 이 원래 잘 그리는 실사풍으로 끌려간다."""
        prop_names = sorted(style_ref.keys() & {pr["name"] for p in pages for panel in p["panels"] for pr in panel["props"]})
        extra_names = sorted(style_ref.keys() - set(prop_names))
        props, extras = {}, {}
        for i, name in enumerate(prop_names):
            with self.timed("소품 그림 (1개, 검사·다시 그리기 포함)"):
                img, sprite = self.drawn_sprite(self._styled(style_ref[name], self.fit_prompt(name, f"{ALONE}, {ASSET_STYLE}"),
                                                             self.negative(NO_CHARACTERS, "background scenery")),
                                                name, self.seed + 500 + i)
            self._save(img, f"prop_{i}.png")
            props[name] = graded(sprite)
        for i, name in enumerate(extra_names):
            with self.timed("조연 그림 (1명, 검사·다시 그리기 포함)"):
                img, sprite = self.drawn_sprite(self._styled(style_ref[name], self.fit_prompt(name, f"full body, {ALONE}, {ASSET_STYLE}"),
                                                             self.negative("background scenery")),
                                                name, self.seed + 700 + i)
            self._save(img, f"extra_{i}.png")
            sprite = graded(sprite)
            extras[name] = {"sprite": sprite, "rig": choose_limbs(find_parts(sprite))}   # 조연은 자세 없이 서 있게
        print("따로 그린 소품:", prop_names, "| 여러 번 나오는 조연:", extra_names, flush=True)
        return props, extras

    def timing_report(self, n_characters, pages):
        lines = [f"모드: 아이 캐릭터 기본 SDXL {self.character_steps}단계, 나머지 "
                 f"{'빠른 SDXL (Hyper-SD 8단계)' if self.fast else '기본 SDXL 30단계'}, 배경 1장 (자리 막히면 최대 {self.bg_tries}장), "
                 f"장면에 녹이기 세기 {self.blend} (안쪽 섞기 {self.inner_mix}), "
                 f"아이 캐릭터 장면 안 다시 그리기 세기 {self.live}, "
                 f"그림체 입히기 {f'켬, 세기 {self.style_live}, 선 {self.style_cn}' if self.style else '끔'}, "
                 f"선 지키기 {'켬' if self.line_keep else '끔'}, 색 지키기 {'켬' if self.color_keep else '끔'}, "
                 f"발광 {'켬' if self.glow else '끔'}, "
                 f"다시 그리기 {'빠른 SDXL' if self.live_fast else '기본 SDXL'}",
                 "단계 | 횟수 | 평균(초) | 합계(초)", "---|---|---|---"]
        for label, ts in self.times.items():
            lines.append(f"{label} | {len(ts)} | {np.mean(ts):.1f} | {np.sum(ts):.1f}")
        total = sum(np.sum(ts) for label, ts in self.times.items()
                    if label != "모델 불러오기 (처음 1번)" and "비교용" not in label
                    and not label.startswith(("장면에 녹이기", "아이 캐릭터 장면 안", "다시 그린 모양")))   # 칸 조립 시간에 이미 들어 있다
        n_panels = sum(len(p.get("panels", [p])) for p in pages)
        lines.append(f"\n→ 동화 1권 (캐릭터 {n_characters}명, {len(pages)}페이지 {n_panels}칸): "
                     f"{total / 60:.1f}분 (모델 불러오기·비교용 제외)")
        total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        lines.append(f"→ GPU 메모리: 최대 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB 사용 / 전체 {total_gb:.1f} GB")
        return "\n".join(lines)


def fetch_isnet(path):
    """배경 제거 모델 파일 준비. Kaggle Input 에 올려 둔 것 -> GitHub(공식) -> Hugging Face 복사본(SHA256 이 맞을 때만) 순.
    2026-11-08 GitHub 내려받기가 504(시간 초과)로 계속 실패해 노트북이 멈췄다 - 전시 날 그림 서버가 같은 일로 못 켜지지 않게."""
    import glob, hashlib, shutil
    if os.path.exists(path):
        return path
    for found in sorted(glob.glob("/kaggle/input/**/isnet-general-use.onnx", recursive=True)):
        shutil.copy(found, path)
        return path
    for url, check in ((ISNET_URL, False), (ISNET_URL, False), (ISNET_MIRROR, True), (ISNET_MIRROR, True)):
        try:
            urllib.request.urlretrieve(url, path + ".part")
            if check:
                h = hashlib.sha256()
                with open(path + ".part", "rb") as f:
                    for chunk in iter(lambda: f.read(1 << 20), b""):
                        h.update(chunk)
                if h.hexdigest() != ISNET_SHA256:
                    raise OSError(f"파일 지문이 다름 ({h.hexdigest()[:12]}...)")
            os.replace(path + ".part", path)
            print(f"배경 제거 모델 받음: {url}", flush=True)
            return path
        except OSError as e:          # HTTPError·URLError·지문 다름 모두
            print(f"배경 제거 모델 내려받기 실패 ({url.split('/')[2]}: {e}) - 10초 뒤 다음 시도", flush=True)
            time.sleep(10)
    raise RuntimeError("배경 제거 모델을 내려받지 못했습니다. isnet-general-use.onnx 를 Kaggle Input 에 올려 주세요.")


def _grow(mask, r):
    """bool 모양을 반지름 r 만큼 넓힌다. 거리 계산이라 r 과 상관없이 빠르다
    (PIL MaxFilter 는 크기의 제곱만큼 느려져 768 그림에서 크기 169 가 21초 걸렸다 - 2026-11-08 칸당 40초의 원인)."""
    return ndimage.distance_transform_edt(~mask) <= r


def _face_marks(before, core):
    """몸 안의 진한 표시 (아이가 그린 눈·입·코 등): 윤곽선에서 떨어진 몸 안쪽 중 몸 색보다 뚜렷이 어두운 곳."""
    S = core.shape[0]
    lum = np.array(before.convert("L")).astype(float)
    inner = ndimage.distance_transform_edt(core) > S * 0.03
    if inner.sum() <= 100:
        return np.zeros_like(core)
    marks = inner & (lum < np.median(lum[inner]) - 50)
    return marks if marks.sum() > 30 else np.zeros_like(core)


def _keep_colors(before, after, body):
    """다시 그린 몸(body)의 색을 원래 색으로 되돌린다. 채널마다 '원래 몸의 가운데 값 / 다시 그린 몸의 가운데 값' 을 곱하므로
    붓 자국·명암(밝고 어두운 차이)은 남고 전체 색조만 돌아온다 (2026-11-08: 배경 그림체를 따라가며 주황 악당이 노랑으로).
    가장자리는 부드럽게 섞어 몸 경계에 색 띠가 생기지 않게 한다."""
    if body.sum() < 100:
        return after
    a0 = np.array(before.convert("RGB")).astype(np.float32)
    a1 = np.array(after.convert("RGB")).astype(np.float32)
    gain = np.clip((np.median(a0[body], axis=0) + 1) / (np.median(a1[body], axis=0) + 1), 0.5, 2.0)
    w = ndimage.gaussian_filter(body.astype(np.float32), 2)[..., None]
    fixed = np.clip(a1 * gain, 0, 255)
    return Image.fromarray((a1 * (1 - w) + fixed * w).astype(np.uint8))


def _drop_new_colors(before, after, body, limit=30):
    """다시 그린 몸 안에서 색조가 원래와 뚜렷이 달라진 얼룩(볼터치·새 무늬)을 원래 그림으로 되돌린다.
    색조 = 픽셀 색에서 밝기(세 채널 평균)를 뺀 것. 명암·붓 자국은 주로 밝기 변화라 남고, 하얀 몸에 생긴 주황 볼처럼
    색 자체가 바뀐 곳만 걸린다 (2026-11-11 앱 시험: 주인공 볼터치는 부드러운 색이라 새 무늬 검사에도 안 걸렸다).
    색 지키기(_keep_colors)로 몸 전체 색조를 맞춘 뒤에 쓴다 - 그래야 남은 차이가 '부분 얼룩'이다."""
    if body.sum() < 100:
        return after
    a0 = np.array(before.convert("RGB")).astype(np.float32)
    a1 = np.array(after.convert("RGB")).astype(np.float32)
    c0 = a0 - a0.mean(axis=2, keepdims=True)
    c1 = a1 - a1.mean(axis=2, keepdims=True)
    changed = body & (np.linalg.norm(c1 - c0, axis=2) > limit)
    if not changed.any():
        return after
    w = ndimage.gaussian_filter((_grow(changed, 5) & body).astype(np.float32), 2)[..., None]   # 얼룩 가장자리 테두리까지
    return Image.fromarray((a1 * (1 - w) + a0 * w).astype(np.uint8))


def _child_lines(before, shape, core):
    """아이가 그린 선: 캐릭터 모양 안에서 몸 색보다 뚜렷이 진한 곳 (윤곽선·막대 팔·눈·입). 몸 자체가 진한 캐릭터는
    몸 색 기준이라 몸 전체가 잡히지 않는다."""
    lum = np.array(before.convert("L")).astype(float)
    if not core.any():
        return np.zeros_like(shape)
    return shape & (lum < np.median(lum[core]) - 50)


def _live_scores(before, after, core, background):
    """장면 안 다시 그리기 검사 점수.
    before/after: 다시 그리기 전·후 자른 그림. core: 원래 몸 자리. background: 같은 자리의 배경(캐릭터 없음).
    - body: 원래 몸 중 배경과 뚜렷이 달랐던 곳이 다시 그린 뒤에도 배경과 다른 비율 (몸이 배경으로 지워지면 낮다).
      배경 제거 모델은 밝은 배경 위 하얀·노란 캐릭터를 놓쳐서(2026-11-08 주인공 0.46, 별 0.02) 쓰지 않는다
    - face: 원래 얼굴 표시가 원래 진하기의 절반 이상으로 남은 비율 (표시가 없으면 1)
    - new: 몸 안에 원래 없던 또렷한 무늬(새 눈·입·이빨)가 생긴 넓이 / 원래 얼굴 표시 넓이 (표시가 작거나 없으면 몸 안쪽의 2%).
      또렷함 = 자기 주변 평균과의 밝기 차이라서, 부드러운 음영·옅은 질감·볼터치는 거의 세지 않는다.
      (2026-11-08: 그림체 입히기에서 악당이 초록 눈 고양이로 바뀌었는데 face 는 0.76 으로 통과 - '남았나'만 봤기 때문)"""
    S = core.shape[0]
    b0 = np.abs(np.array(before).astype(float) - np.array(background).astype(float)).sum(axis=2)
    b1 = np.abs(np.array(after).astype(float) - np.array(background).astype(float)).sum(axis=2)
    stood_out = core & (b0 > 60)
    body = float((b1[stood_out] > 0.4 * b0[stood_out]).mean()) if stood_out.sum() > 100 else 1.0
    lum0 = np.array(before.convert("L")).astype(float)
    lum1 = np.array(after.convert("L")).astype(float)
    inner = ndimage.distance_transform_edt(core) > S * 0.03
    face = 1.0
    marks = _face_marks(before, core)
    if marks.any():
        darkest = ndimage.minimum_filter(lum1, size=max(3, S // 100) | 1)   # 표시가 몇 픽셀 움직여도 남은 것으로
        was = np.median(lum0[inner]) - lum0[marks]
        now = np.median(lum1[inner]) - darkest[marks]
        face = float((now >= 0.5 * was).mean())
    d0 = np.abs(lum0 - ndimage.gaussian_filter(lum0, 3))
    d1 = np.abs(lum1 - ndimage.gaussian_filter(lum1, 3))
    fresh = inner & (d1 > 25) & ~_grow(d0 > 15, S * 0.01)
    new_detail = float(fresh.sum() / max(int(marks.sum()), int(inner.sum() * 0.02), 1))
    return {"body": body, "face": face, "new": new_detail}


def _safe(name):
    return "".join(c if c.isalnum() else "_" for c in name)[:40]


def _on_white(img):
    """투명 바탕 그림(앱에서 오는 PNG)을 흰 바탕 512x512 로."""
    img = img.convert("RGBA")
    return Image.alpha_composite(Image.new("RGBA", img.size, "white"), img).convert("RGB").resize((512, 512))

In [ ]:
%%writefile image_server.py
"""Kaggle 그림 서버: 기존 서버(storybook_server)가 동화 한 권을 맡기면 우리 모델로 그려서 돌려준다.

    POST /jobs           {"characters": {id: {"image_b64", "desc", "arms", "legs"}}, "plan": {"pages": [...]}}
                         -> {"job_id": "..."}
    POST /characters     {"characters": {...}} -> {"job_id": "..."}   캐릭터 완성만 미리 (그림 결과는 돌려주지 않고 캐시에 둔다).
                         아이가 그림을 다 그렸을 때 기존 서버가 보낸다 -> 나중에 동화를 맡기면 같은 그림·설명은 다시 그리지 않는다
    GET  /jobs/{job_id}  -> {"status": "queued" | "running" | "done" | "error", "pages": [{"page", "image_b64"}], "error"}
    GET  /health         -> {"ok": true, "queued": n}

모든 요청은 헤더 X-Token 이 맞아야 한다 (터널 주소는 인터넷에 열려 있으므로).
Cloudflare 임시 터널은 응답을 100초까지만 기다려서, 동화 한 권(약 5분)은 '맡기고 끝났는지 묻는' 방식으로 주고받는다.
GPU 는 한 번에 한 권만 그린다 - 뒤에 온 요청은 줄을 선다.
"""
import base64
import io
import queue
import threading
import time
import traceback
import uuid

from fastapi import FastAPI, Header, HTTPException
from PIL import Image

KEEP_DONE = 20      # 끝난 작업 결과는 최근 것만 들고 있는다 (메모리)


def _decode(b64):
    if "," in b64:
        b64 = b64.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64)))


def _encode(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def create_app(pipe, token):
    app = FastAPI()
    jobs = {}
    waiting = queue.Queue()

    def check(x_token):
        if x_token != token:
            raise HTTPException(status_code=401, detail="token")

    def worker():
        while True:
            job_id, body = waiting.get()
            job = jobs[job_id]
            job["status"], t0 = "running", time.perf_counter()
            try:
                characters = {k: dict(v, image=_decode(v["image_b64"])) for k, v in body["characters"].items()}
                assets = pipe.make_characters(characters)
                if "plan" not in body:                  # 캐릭터 미리 그리기만
                    job["status"] = "done"
                    print(f"✅ 작업 {job_id[:8]} 캐릭터 미리 그리기 완료: {list(characters)}, "
                          f"{time.perf_counter() - t0:.0f}초", flush=True)
                    continue
                pictures = pipe.make_book(body["plan"], assets)
                numbers = [p.get("page", i + 1) for i, p in enumerate(body["plan"].get("pages", []))]
                job["pages"] = [{"page": n, "image_b64": _encode(img)} for n, img in zip(numbers, pictures)]
                job["status"] = "done"
                print(f"✅ 작업 {job_id[:8]} 완료: {len(pictures)}페이지, {time.perf_counter() - t0:.0f}초", flush=True)
            except Exception as e:
                job["status"], job["error"] = "error", f"{type(e).__name__}: {e}"
                traceback.print_exc()
            finally:
                done =[k for k, j in jobs.items() if j["status"] in ("done", "error")]
                for k in done[:-KEEP_DONE]:
                    jobs.pop(k, None)

    threading.Thread(target=worker, daemon=True).start()

    @app.get("/health")
    def health(x_token: str = Header(default="")):
        check(x_token)
        return {"ok": True, "queued": waiting.qsize()}

    @app.post("/jobs")
    def submit(body: dict, x_token: str = Header(default="")):
        check(x_token)
        if not body.get("plan", {}).get("pages") or not isinstance(body.get("characters"), dict):
            raise HTTPException(status_code=400, detail="plan.pages 와 characters 가 필요합니다")
        job_id = uuid.uuid4().hex
        jobs[job_id] = {"status": "queued", "pages": [], "error": None}
        waiting.put((job_id, body))
        print(f"📥 작업 {job_id[:8]} 접수: {len(body['plan']['pages'])}페이지, 캐릭터 {list(body['characters'])}", flush=True)
        return {"job_id": job_id}

    @app.post("/characters")
    def prepare(body: dict, x_token: str = Header(default="")):
        check(x_token)
        chars = body.get("characters")
        if not isinstance(chars, dict) or not chars:
            raise HTTPException(status_code=400, detail="characters 가 필요합니다")
        job_id = uuid.uuid4().hex
        jobs[job_id] = {"status": "queued", "pages": [], "error": None}
        waiting.put((job_id, {"characters": chars}))   # 동화와 같은 줄 - GPU 는 한 번에 하나, 뒤에 온 동화는 이것을 기다렸다가 캐시를 쓴다
        print(f"📥 작업 {job_id[:8]} 접수: 캐릭터 미리 그리기 {list(chars)}", flush=True)
        return {"job_id": job_id}

    @app.get("/jobs/{job_id}")
    def status(job_id: str, x_token: str = Header(default="")):
        check(x_token)
        if job_id not in jobs:
            raise HTTPException(status_code=404, detail="job")
        return jobs[job_id]

    return app

In [ ]:
# ② 모델 불러오기 (한 번만)
import sys

sys.path.insert(0, "/kaggle/working")
from story_pipeline import StoryPipeline

SAVE_DEBUG = True    # 칸마다 배경 후보·캐릭터 원본 등을 /kaggle/working/debug 에 저장 (시험 기간에는 켜 둔다, 전시 때는 False)
pipe = StoryPipeline(save_dir="/kaggle/working/debug" if SAVE_DEBUG else None).load()

In [ ]:
# ③ 서버 켜기 + 터널 주소 받기
import re
import secrets
import subprocess
import threading

import uvicorn
from image_server import create_app

TOKEN = secrets.token_urlsafe(24)    # 터널 주소는 인터넷에 열려 있으므로, 이 값을 아는 서버만 그림을 맡길 수 있다
server = uvicorn.Server(uvicorn.Config(create_app(pipe, TOKEN), host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()

tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0)
        break
# 터널 로그를 계속 비워 줘야 터널이 멈추지 않는다
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

print("아래 두 줄을 storybook_server/.env 에 붙이고 서버를 다시 켜세요 (앱에서 '우리 모델'을 고르면 이 서버로 그립니다):\n")
print(f"LOCAL_IMAGE_URL={URL}")
print(f"LOCAL_IMAGE_TOKEN={TOKEN}")

In [ ]:
# ④ 계속 켜 두기 - 이 셀이 도는 동안 서버가 산다. 1분마다 살아 있다고 알린다
import time

while True:
    time.sleep(60)
    print(time.strftime("%H:%M"), "그림 서버 동작 중", flush=True)

In [ ]:
# ⑤ (문제 볼 때) 중간 그림 내려받기 - ④ 셀을 멈춘(■) 뒤 이 셀을 실행한다. ④를 멈춰도 서버는 계속 켜져 있다
# 같은 이름 파일은 다음 동화가 덮어쓰므로, 본 동화 바로 뒤에 받는다
import os

from IPython.display import FileLink

os.chdir("/kaggle/working")
!zip -qr debug.zip debug
FileLink("debug.zip")